Phase A — 입력값 (의존성 없음, 이미 완료)

Cell 0: 펀드 데이터, 현재 포트폴리오, 외부자산
Cell 1: rf_rate, borrowing_cost
Cell 2-3: Sharpe, AR-adjusted metrics
Cell 4-6: 12×12 correlation matrix


Phase B — 현재 포트폴리오 진단 (A만 필요)
Cell 7: 현재 포트폴리오의 return, vol, Sharpe, 연간 income 추정. 이게 있어야 "지금 뭐가 문제인지"를 숫자로 보여줄 수 있고, 나중에 제안 포트폴리오와 비교할 기준이 돼.

Phase C — SAA 결정 (A + B 필요)
Cell 8: 여기서 비중이 정해져야 해. MVO든 팀 판단이든. 이게 전체의 병목이야 — 이 숫자가 나와야 뒤에 모든 게 돌아가.

Phase D — SAA가 나온 뒤 (A + C 필요)
이 5개는 전부 SAA 비중에 의존하지만 서로는 독립이라 순서 상관없어:
Cell 9: 제안 포트폴리오 return, vol, Sharpe + 현재 대비 개선 비교
Cell 10: Cash return vs Capital gain 분리 — asset class별 yield 가정을 넣어서 income 부분과 시세차익 부분을 나눔. 케이스가 명시적으로 요구한 항목
Cell 11: Income sufficiency — 연 $300K 목표 충족 여부 + 교육비 projection (쌍둥이 13년 뒤 미국 대학)
Cell 12: Fund selection — SAA 비중에 따라 각 class에서 어떤 펀드를 고르는지 + 근거
Cell 13: Credit solution — $80M 투자 구조. $45M 현금 + 외부담보 차입 + 펀드 AR 레버리지. 이자 비용, net return

Phase E — 시각화 (D 필요)
Cell 14: Efficient frontier (현재 + 제안 포트폴리오 위치 표시), correlation heatmap, current vs proposed bar chart. 전부 Deliverable 3 교육 슬라이드용.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)

# ═══════════════════════════════════════════════════════════════
# APPENDIX I: Traditional Funds (44 funds)
# ═══════════════════════════════════════════════════════════════

app1_data = {
    "ISIN": [
        "LU0079474960","LU0823410997","LU0210528500","LU0210536511","LU0674140396",
        "LU1153584641","LU0171276677","LU0997586606","LU1193295406","LU0336375786",
        "LU0006061252","LU0129465034","LU1684384271","LU0936264273","LU1453624402",
        "IE00B067MR52","LU0197773160","HK0000055555","LU0188438112","LU0390135332",
        "LU1728044204","LU1128926489","IE00B8K7V925","LU1514167136",
        "LU1008669860","LU0171284770","LU0780247044","LU0418832605",
        "LU0200680600","LU0605512275","LU0562246024",
        "LU0679941327","LU0780247804","LU0692309627",
        "LU1548497426","LU0124384867","LU0106831901","LU0055631609","LU0109394709",
        "IE0009355771","LU0070992663","LU0101692670","LU1861558580","LU1983299162"
    ],
    "Fund": [
        "AB American Growth Portfolio","BNP US Small Cap","JPMorgan America Equity Fund",
        "JPMorgan US Value","Robeco US Select Opportunities Equity",
        "BlackRock European Equity Income","BlackRock European Special Situations Fund",
        "Fidelity European Growth Fund","HSBC GIF Euroland Value Equity","JPMorgan Europe Equity Plus",
        "BlackRock Japan Small & Midcap Opportunities","JPMorgan Japan","M&G Japan",
        "Pictet Japanese Equity Opportunities","Schroder Japanese Equity",
        "FSSA Asian Equity Plus","HSBC Asia ex Japan Equity High Dividend","JPMorgan ASEAN",
        "Schroder Asian Equity Yield","Templeton Asian Smaller Companies",
        "HSBC Global IG Securitised Credit","JPMorgan Income","PIMCO Income","Schroder Global Credit Income",
        "AB Global High Yield","BlackRock Global High Yield Bond","HSBC Global High Yield Bond","Schroder Global High Yield",
        "BlackRock Emerging Markets Bond","Fidelity Asian Bond","JPMorgan EM Investment Grade",
        "BlackRock China Bond","HSBC GIF India Fixed Income","HSBC GIF RMB Fixed Income",
        "Allianz Global AI","BlackRock Sustainable Energy","BlackRock World Financials",
        "BlackRock World Gold","Franklin Biotechnology Discovery","Janus Henderson Global Life Sciences",
        "Janus Henderson Global Technology Leaders","Pictet Digital",
        "Nikko AM ARK Disruptive Innovation","Schroder Global Alternative Energy"
    ],
    "Asset Class": [
        "US Equities","US Equities","US Equities","US Equities","US Equities",
        "European Equities","European Equities","European Equities","European Equities","European Equities",
        "Japanese Equities","Japanese Equities","Japanese Equities","Japanese Equities","Japanese Equities",
        "Asia ex-Japan Equities","Asia ex-Japan Equities","Asia ex-Japan Equities","Asia ex-Japan Equities","Asia ex-Japan Equities",
        "Global IG Bonds","Global IG Bonds","Global IG Bonds","Global IG Bonds",
        "Global HY Bonds","Global HY Bonds","Global HY Bonds","Global HY Bonds",
        "EM Hard Currency Debt","EM Hard Currency Debt","EM Hard Currency Debt",
        "EM Local Currency Debt","EM Local Currency Debt","EM Local Currency Debt",
        "Thematic","Thematic","Thematic","Thematic","Thematic","Thematic","Thematic","Thematic","Thematic","Thematic"
    ],
    "Return": [
        0.126,0.080,0.144,0.106,0.107,
        0.107,0.022,0.101,0.165,0.157,
        0.051,0.037,0.126,0.180,0.036,
        0.026,0.077,0.058,0.087,0.088,
        0.035,0.023,0.032,0.024,
        0.039,0.034,0.024,0.038,
        0.032,0.010,-0.002,
        0.014,0.004,0.004,
        0.070,0.071,0.191,0.145,0.052,0.056,0.158,0.032,-0.027,-0.005
    ],
    "Volatility": [
        0.182,0.204,0.156,0.148,0.180,
        0.130,0.206,0.122,0.167,0.162,
        0.143,0.194,0.143,0.124,0.141,
        0.158,0.160,0.145,0.167,0.155,
        0.020,0.043,0.057,0.063,
        0.073,0.067,0.066,0.069,
        0.098,0.098,0.075,
        0.107,0.050,0.060,
        0.257,0.212,0.242,0.305,0.199,0.154,0.206,0.057,0.420,0.263
    ],
    "AR": [
        0.70,0.70,0.70,0.70,0.70,
        0.70,0.70,0.70,0.70,0.70,
        0.70,0.70,0.70,0.70,0.70,
        0.70,0.70,0.70,0.70,0.70,
        0.80,0.70,0.80,0.80,
        0.60,0.70,0.60,0.70,
        0.75,0.75,0.80,
        0.75,0.75,0.80,
        0.70,0.70,0.70,0.70,0.70,0.70,0.75,0.70,0.70,0.70
    ]
}

funds = pd.DataFrame(app1_data).set_index("ISIN")

# ═══════════════════════════════════════════════════════════════
# APPENDIX II: Alternative Funds (9 funds)
# ═══════════════════════════════════════════════════════════════

app2_data = {
    "Fund": [
        "Global Private Equity Fund","Global Private Credit Fund",
        "Global Infrastructure Income Fund","US Core Plus Real Estate Income Fund",
        "Global Multi-Strategy Multi-PM","Global Equity L/S","Asia Equity L/S",
        "Global Macro","Global Fund of Hedge Funds"
    ],
    "Type": [
        "Private Markets","Private Markets","Private Markets","Private Markets",
        "Hedge Fund","Hedge Fund","Hedge Fund","Hedge Fund","Hedge Fund"
    ],
    "Correlation": [0.17, 0.35, 0.52, 0.56, 0.21, 0.29, 0.33, -0.24, 0.66],
    "Benchmark": [
        "MSCI World","ICE BofA US High Yield","S&P Global Infra. Index","D.J. US Real Estate",
        "MSCI World","MSCI World","MSCI Asia Pacific","MSCI World","MSCI World"
    ],
    "Return": [0.099, 0.082, 0.080, 0.089, 0.103, 0.144, 0.196, 0.094, 0.046],
    "Volatility": [0.196, 0.136, 0.110, 0.113, 0.068, 0.101, 0.107, 0.058, 0.033],
    "AR": [0.50, 0.50, 0.50, 0.50, 0.40, 0.40, 0.40, 0.40, 0.40],
    "Min Investment": [150000,150000,150000,150000,250000,250000,250000,250000,250000],
    "Max Quarterly Redemption": [0.125]*9
}

alts = pd.DataFrame(app2_data)

# ═══════════════════════════════════════════════════════════════
# CLIENT DATA
# ═══════════════════════════════════════════════════════════════

# Current portfolio (as stated in case)
current_portfolio_raw = pd.DataFrame({
    "Asset Class": ["US Equities","HK Equities","SG Equities","Global HY Bonds","Cash"],
    "Weight": [0.70, 0.05, 0.03, 0.02, 0.20]
})

# Mapped to correlation matrix classes (HK + SG → Asia ex-JP Eq)
current_portfolio = pd.DataFrame({
    "Asset Class": ["US Eq","Asia ex-JP Eq","Global HY","Cash"],
    "Weight":      [0.70,   0.08,           0.02,       0.20],
    "Note":        ["US Equities 70%",
                    "HK Equities 5% + SG Equities 3%",
                    "Global High Yield 2%",
                    "Cash 20%"]
})

# External assets (for credit solution)
external_assets = pd.DataFrame({
    "Asset": ["LionDash Shares","Singapore Commercial Property","Fine Art"],
    "Value": [30_000_000, 15_000_000, 5_000_000],
    "Standard AR": [0.50, 0.50, 0.75]
})

investable_aum = 45_000_000
target_investment = 80_000_000
monthly_expense = 50_000
income_target = monthly_expense * 0.5 * 12   # $300K/yr

# ═══════════════════════════════════════════════════════════════
# GOVT BONDS PROXY (in corr matrix but no Appendix fund)
# ═══════════════════════════════════════════════════════════════

govt_bond_proxy = {
    "Return": 0.035,    # ~risk-free rate
    "Volatility": 0.065 # IEF 5yr approx
}

# ═══════════════════════════════════════════════════════════════
# YIELD ASSUMPTIONS (for cash return vs capital gain split)
# Total Return = Yield (cash return) + Capital Gain
# ═══════════════════════════════════════════════════════════════

yield_assumptions = pd.DataFrame({
    "Asset Class": [
        "US Eq","EU Eq","JP Eq","Asia ex-JP Eq",
        "Global IG","Global HY","EM HC Debt","EM LC Debt",
        "Govt Bonds","Cash",
        "Private Markets","Hedge Funds"
    ],
    "Est Yield": [
        0.005, 0.030, 0.020, 0.025,
        0.040, 0.065, 0.055, 0.050,
        0.030, 0.035,
        0.045, 0.010
    ],
    "Source": [
        "S&P 500 avg dividend yield",
        "Euro Stoxx avg dividend yield",
        "TOPIX avg dividend yield",
        "MSCI Asia ex-JP avg dividend yield",
        "IG avg coupon",
        "HY avg coupon",
        "EM HC avg coupon",
        "EM LC avg coupon",
        "US 5-10yr treasury yield",
        "Money market rate = rf",
        "PE/RE distribution yield",
        "HF: minimal yield, mostly capital gain"
    ]
})

# ═══════════════════════════════════════════════════════════════
# VALIDATION
# ═══════════════════════════════════════════════════════════════

print("Data loaded successfully.")
print(f"Appendix I:  {len(funds)} funds across {funds['Asset Class'].nunique()} asset classes")
print(f"Appendix II: {len(alts)} alternative funds")
print(f"\nClient: investable ${investable_aum/1e6:.0f}M, target investment ${target_investment/1e6:.0f}M")
print(f"Income target: ${income_target:,.0f}/yr")
print(f"\nCurrent portfolio (mapped):")
print(current_portfolio[["Asset Class","Weight"]].to_string(index=False))
print(f"\nYield assumptions loaded: {len(yield_assumptions)} asset classes")

Data loaded successfully.
Appendix I:  44 funds across 9 asset classes
Appendix II: 9 alternative funds

Client: investable $45M, target investment $80M
Income target: $300,000/yr

Current portfolio (mapped):
  Asset Class  Weight
        US Eq    0.70
Asia ex-JP Eq    0.08
    Global HY    0.02
         Cash    0.20

Yield assumptions loaded: 12 asset classes


In [2]:
rf_rate = 0.035
borrowing_cost = 0.05

In [3]:
funds["Sharpe"] = (funds["Return"] - rf_rate) / funds["Volatility"]
funds["AR Adj Return"] = (funds["Return"] - funds["AR"] * borrowing_cost) / (1 - funds["AR"])
funds["AR Adj Vol"] = funds["Volatility"] / (1 - funds["AR"])
funds["AR Adj Sharpe"] = (funds["AR Adj Return"] - rf_rate) / funds["AR Adj Vol"]

for ac in funds["Asset Class"].unique():
    subset = funds[funds["Asset Class"] == ac].sort_values("Sharpe", ascending=False)
    print(f"\n▸ {ac}")
    print(subset[["Fund","Return","Volatility","AR","Sharpe","AR Adj Return","AR Adj Sharpe"]].to_string())


▸ US Equities
                                               Fund  Return  Volatility   AR    Sharpe  AR Adj Return  AR Adj Sharpe
ISIN                                                                                                                
LU0210528500           JPMorgan America Equity Fund   0.144       0.156  0.7  0.698718       0.363333       0.631410
LU0079474960           AB American Growth Portfolio   0.126       0.182  0.7  0.500000       0.303333       0.442308
LU0210536511                      JPMorgan US Value   0.106       0.148  0.7  0.479730       0.236667       0.408784
LU0674140396  Robeco US Select Opportunities Equity   0.107       0.180  0.7  0.400000       0.240000       0.341667
LU0823410997                       BNP US Small Cap   0.080       0.204  0.7  0.220588       0.150000       0.169118

▸ European Equities
                                                    Fund  Return  Volatility   AR    Sharpe  AR Adj Return  AR Adj Sharpe
ISIN                   

In [4]:
alts["Sharpe"] = (alts["Return"] - rf_rate) / alts["Volatility"]
alts["AR Adj Return"] = (alts["Return"] - alts["AR"] * borrowing_cost) / (1 - alts["AR"])
alts["AR Adj Vol"] = alts["Volatility"] / (1 - alts["AR"])
alts["AR Adj Sharpe"] = (alts["AR Adj Return"] - rf_rate) / alts["AR Adj Vol"]

print(alts[["Fund","Type","Return","Volatility","AR","Correlation","Sharpe","AR Adj Return","AR Adj Sharpe"]].sort_values("Sharpe", ascending=False).to_string(index=False))

                                Fund            Type  Return  Volatility  AR  Correlation   Sharpe  AR Adj Return  AR Adj Sharpe
                     Asia Equity L/S      Hedge Fund   0.196       0.107 0.4         0.33 1.504673       0.293333       1.448598
                   Global Equity L/S      Hedge Fund   0.144       0.101 0.4         0.29 1.079208       0.206667       1.019802
                        Global Macro      Hedge Fund   0.094       0.058 0.4        -0.24 1.017241       0.123333       0.913793
      Global Multi-Strategy Multi-PM      Hedge Fund   0.103       0.068 0.4         0.21 1.000000       0.138333       0.911765
US Core Plus Real Estate Income Fund Private Markets   0.089       0.113 0.5         0.56 0.477876       0.128000       0.411504
   Global Infrastructure Income Fund Private Markets   0.080       0.110 0.5         0.52 0.409091       0.110000       0.340909
          Global Private Credit Fund Private Markets   0.082       0.136 0.5         0.35 0.34558

In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL: Unified Fund Metrics Table (53 funds)
# ═══════════════════════════════════════════════════════════════
# Dependencies: Cell 0 (funds, alts, yield_assumptions), Cell 1 (rf_rate, borrowing_cost)
# Re-run after changing rf_rate or borrowing_cost
# ═══════════════════════════════════════════════════════════════

# ── 1. Prepare Appendix I (44 traditional funds) ─────────────

df1 = funds.copy()
df1["Source"] = "Appendix I"
df1["Correlation"] = np.nan
df1["Benchmark"] = np.nan
df1["Min Investment"] = np.nan
df1["Max Quarterly Redemption"] = np.nan

# ── 2. Prepare Appendix II (9 alternative funds) ─────────────

df2 = alts.copy()
df2.index.name = "ISIN"
df2["ISIN"] = ["ALT_" + str(i+1).zfill(2) for i in range(len(df2))]
df2 = df2.set_index("ISIN")
df2 = df2.rename(columns={"Type": "Asset Class"})
df2["Source"] = "Appendix II"

# ── 3. Combine ────────────────────────────────────────────────

all_funds = pd.concat([df1, df2], sort=False)

# ── 4. Yield assumptions (asset class level) ──────────────────

yield_map = {
    "US Equities": 0.005, "European Equities": 0.030, "Japanese Equities": 0.020,
    "Asia ex-Japan Equities": 0.025, "Global IG Bonds": 0.040, "Global HY Bonds": 0.065,
    "EM Hard Currency Debt": 0.055, "EM Local Currency Debt": 0.050,
    "Thematic": 0.005, "Private Markets": 0.045, "Hedge Fund": 0.010,
}
all_funds["Est Yield"] = all_funds["Asset Class"].map(yield_map)

# ── 5. Calculations ───────────────────────────────────────────

# Existing metrics
all_funds["Sharpe"] = (all_funds["Return"] - rf_rate) / all_funds["Volatility"]
all_funds["AR Adj Return"] = (all_funds["Return"] - all_funds["AR"] * borrowing_cost) / (1 - all_funds["AR"])
all_funds["AR Adj Vol"] = all_funds["Volatility"] / (1 - all_funds["AR"])
all_funds["AR Adj Sharpe"] = (all_funds["AR Adj Return"] - rf_rate) / all_funds["AR Adj Vol"]

# New metrics
all_funds["Break-even Return"] = all_funds["AR"] * borrowing_cost
all_funds["Excess over BE"] = all_funds["Return"] - all_funds["Break-even Return"]
all_funds["Leveraged Exposure"] = 1 / (1 - all_funds["AR"])
all_funds["Leveraged Income (per $1)"] = all_funds["Est Yield"] * all_funds["Leveraged Exposure"]

# ── 6. Display ────────────────────────────────────────────────

display_cols = [
    "Fund", "Asset Class", "Source",
    "Return", "Volatility", "AR", "Est Yield",
    "Sharpe", "AR Adj Return", "AR Adj Vol", "AR Adj Sharpe",
    "Break-even Return", "Excess over BE",
    "Leveraged Exposure", "Leveraged Income (per $1)",
    "Correlation", "Benchmark", "Min Investment", "Max Quarterly Redemption",
]

# Sort by asset class then Sharpe descending
output = all_funds[display_cols].sort_values(["Asset Class", "Sharpe"], ascending=[True, False])

print(f"Total funds: {len(output)}")
print(f"Assumptions: rf = {rf_rate:.2%}, borrowing cost = {borrowing_cost:.2%}")
print(f"{'='*100}")

for ac in output["Asset Class"].unique():
    subset = output[output["Asset Class"] == ac]
    print(f"\n▸ {ac} ({len(subset)} funds)")
    print(subset.to_string(index=True))
    print()

Total funds: 53
Assumptions: rf = 3.50%, borrowing cost = 5.00%

▸ Asia ex-Japan Equities (5 funds)
                                                 Fund             Asset Class      Source  Return  Volatility   AR  Est Yield    Sharpe  AR Adj Return  AR Adj Vol  AR Adj Sharpe  Break-even Return  Excess over BE  Leveraged Exposure  Leveraged Income (per $1)  Correlation Benchmark  Min Investment  Max Quarterly Redemption
ISIN                                                                                                                                                                                                                                                                                                                                
LU0390135332        Templeton Asian Smaller Companies  Asia ex-Japan Equities  Appendix I   0.088       0.155  0.7      0.025  0.341935       0.176667    0.516667       0.274194              0.035           0.053            3.333333                  

In [6]:
import wrds

db = wrds.Connection()

tickers = {
    'SPY':  'US Eq',
    'VGK':  'EU Eq',
    'EWJ':  'JP Eq',
    'AAXJ': 'Asia ex-JP Eq',
    'LQD':  'Global IG',
    'HYG':  'Global HY',
    'EMB':  'EM HC Debt',
    'EMLC': 'EM LC Debt',
    'IEF':  'Govt Bonds',
    'BIL':  'Cash',
}

ticker_list = "', '".join(tickers.keys())

sql = f"""
    SELECT a.date, a.ret, b.ticker
    FROM crsp.msf a
    JOIN crsp.msenames b
        ON a.permno = b.permno
        AND b.namedt <= a.date
        AND a.date <= b.nameendt
    WHERE b.ticker IN ('{ticker_list}')
        AND a.date >= '2021-02-01'
        AND a.date <= '2026-01-31'
    ORDER BY a.date, b.ticker
"""

raw = db.raw_sql(sql)
db.close()

print(f"Rows fetched: {len(raw)}")
print(f"Tickers found: {sorted(raw['ticker'].unique())}")
print(f"Date range: {raw['date'].min()} to {raw['date'].max()}")

# Check if any tickers are missing
found = set(raw['ticker'].unique())
missing = set(tickers.keys()) - found
if missing:
    print(f"\n⚠ Missing tickers: {missing}")
    print("These may need alternative proxies.")
else:
    print("\n✓ All tickers found.")

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Rows fetched: 470
Tickers found: ['AAXJ', 'BIL', 'EMB', 'EMLC', 'EWJ', 'HYG', 'IEF', 'LQD', 'SPY', 'VGK']
Date range: 2021-02-26 to 2024-12-31

✓ All tickers found.


In [7]:
pivot = raw.pivot_table(index='date', columns='ticker', values='ret')
pivot = pivot.rename(columns=tickers)
pivot = pivot.dropna()

print(f"Monthly observations: {len(pivot)}")
print(f"Date range: {pivot.index.min()} to {pivot.index.max()}")
print()

corr_wrds = pivot.corr()

print("Correlation Matrix (WRDS / CRSP, monthly returns)")
print("=" * 80)
print(corr_wrds.round(2).to_string())

eigenvalues_w = np.linalg.eigvalsh(corr_wrds.values)
print(f"\nMatrix validation: {'✓ Positive semi-definite' if all(eigenvalues_w >= -1e-10) else '✗ NOT valid'}")
print(f"Smallest eigenvalue: {eigenvalues_w.min():.6f}")

Monthly observations: 47
Date range: 2021-02-26 to 2024-12-31

Correlation Matrix (WRDS / CRSP, monthly returns)
ticker         Asia ex-JP Eq  Cash  EM HC Debt  EM LC Debt  JP Eq  Global HY  Govt Bonds  Global IG  US Eq  EU Eq
ticker                                                                                                           
Asia ex-JP Eq           1.00  0.24        0.77        0.77   0.69       0.53        0.58       0.69   0.53   0.68
Cash                    0.24  1.00        0.28        0.29   0.22       0.19        0.17       0.20   0.09   0.09
EM HC Debt              0.77  0.28        1.00        0.89   0.85       0.84        0.78       0.91   0.80   0.86
EM LC Debt              0.77  0.29        0.89        1.00   0.74       0.66        0.71       0.81   0.63   0.79
JP Eq                   0.69  0.22        0.85        0.74   1.00       0.84        0.71       0.82   0.77   0.80
Global HY               0.53  0.19        0.84        0.66   0.84       1.00        0.72 

In [ ]:
# ── Expand WRDS 10x10 to 12x12 by adding Private Markets & Hedge Funds ──

# Reorder WRDS matrix to match our standard ordering
ordered = ["US Eq","EU Eq","JP Eq","Asia ex-JP Eq",
           "Global IG","Global HY","EM HC Debt","EM LC Debt",
           "Govt Bonds","Cash"]
corr_10 = corr_wrds.loc[ordered, ordered].values

# Appendix II provides correlations vs benchmarks (mostly MSCI World ≈ US Eq proxy)
# Private Markets avg benchmark corr: (0.17+0.35+0.52+0.56)/4 = 0.40
# Hedge Funds avg benchmark corr: (0.21+0.29+0.33-0.24+0.66)/5 = 0.25

# PM and HF correlations with each traditional asset class
# Derived from: benchmark correlation × traditional asset's correlation with that benchmark
pm_corr = np.array([0.40, 0.38, 0.35, 0.35, 0.20, 0.35, 0.30, 0.30, 0.10, 0.05])
hf_corr = np.array([0.25, 0.24, 0.22, 0.20, 0.15, 0.30, 0.20, 0.18, 0.05, 0.03])

# PM vs HF: moderate positive (both are alternative risk premia)
pm_hf = 0.45

# Assemble 12x12
corr_12 = np.zeros((12, 12))
corr_12[:10, :10] = corr_10
corr_12[10, :10] = pm_corr
corr_12[:10, 10] = pm_corr
corr_12[11, :10] = hf_corr
corr_12[:10, 11] = hf_corr
corr_12[10, 11] = pm_hf
corr_12[11, 10] = pm_hf
corr_12[10, 10] = 1.0
corr_12[11, 11] = 1.0

asset_classes_12 = ordered + ["Private Markets", "Hedge Funds"]
corr_final = pd.DataFrame(corr_12, index=asset_classes_12, columns=asset_classes_12)

print("Final 12x12 Correlation Matrix (WRDS + Appendix II Hybrid)")
print("=" * 80)
print(corr_final.round(2).to_string())

eigenvalues_12 = np.linalg.eigvalsh(corr_12)
print(f"\nMatrix validation: {'✓ Positive semi-definite' if all(eigenvalues_12 >= -1e-10) else '✗ NOT valid'}")
print(f"Smallest eigenvalue: {eigenvalues_12.min():.6f}")

print("\n── Source breakdown ──")
print("Rows/Cols 1-10: WRDS/CRSP actual data (2021.02-2024.12, monthly returns)")
print("Rows/Cols 11-12: Estimated from Appendix II benchmark correlations + academic literature")

NameError: name 'corr_wrds' is not defined

In [9]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: MVO → Strategic Asset Allocation (SAA)
# ═══════════════════════════════════════════════════════════════
# Adjust PARAMETERS section and re-run for different scenarios
# Thematic funds excluded from core SAA (satellite allocation)
# ═══════════════════════════════════════════════════════════════

from scipy.optimize import minimize

# ── PARAMETERS (modify and re-run) ─────────────────────────

risk_aversion = 7     # lambda: higher = more conservative (1=aggressive, 3=moderate, 7+=conservative)
min_weight = 0.00         # min weight per asset class
max_weight = 0.40         # max weight per asset class
min_cash   = 0.05         # min cash allocation
income_target = 300_000   # annual income target (USD)
investable_aum = 45_000_000  # investable assets

# Group constraints (equity / FI / alts weight ranges)
equity_range = (0.30, 0.70)
fi_range     = (0.10, 0.40)
alts_range   = (0.05, 0.25)

# ── 1. Representative fund per asset class (Sharpe #1) ─────

class_map = {
    "US Equities":           "US Eq",
    "European Equities":     "EU Eq",
    "Japanese Equities":     "JP Eq",
    "Asia ex-Japan Equities":"Asia ex-JP Eq",
    "Global IG Bonds":       "Global IG",
    "Global HY Bonds":       "Global HY",
    "EM Hard Currency Debt":  "EM HC Debt",
    "EM Local Currency Debt": "EM LC Debt",
}

# Traditional: top Sharpe fund per class
rep_trad = (funds[funds["Asset Class"] != "Thematic"]
            .groupby("Asset Class")
            .apply(lambda x: x.loc[x["Sharpe"].idxmax()])
            [["Fund","Return","Volatility","Sharpe"]])
rep_trad.index = rep_trad.index.map(class_map)

# Alternatives: top Sharpe for PM / HF each
best_pm = alts[alts["Type"]=="Private Markets"].sort_values("Sharpe",ascending=False).iloc[0]
best_hf = alts[alts["Type"]=="Hedge Fund"].sort_values("Sharpe",ascending=False).iloc[0]

rep_alts = pd.DataFrame({
    "Fund":       [best_pm["Fund"], best_hf["Fund"]],
    "Return":     [best_pm["Return"], best_hf["Return"]],
    "Volatility": [best_pm["Volatility"], best_hf["Volatility"]],
    "Sharpe":     [best_pm["Sharpe"], best_hf["Sharpe"]],
}, index=["Private Markets","Hedge Funds"])

# Cash
rep_cash = pd.DataFrame({
    "Fund": ["Cash"], "Return": [rf_rate], "Volatility": [0.001], "Sharpe": [0.0],
}, index=["Cash"])

# Combine
rep = pd.concat([rep_trad, rep_alts, rep_cash])

print("▸ Representative funds per asset class (Sharpe #1)")
print(rep[["Fund","Return","Volatility","Sharpe"]].to_string())

# ── 2. Covariance matrix ───────────────────────────────────

saa_classes = list(rep.index)
mu = rep["Return"].values
vol = rep["Volatility"].values

corr_sub = corr_final.loc[saa_classes, saa_classes].values
cov_matrix = np.diag(vol) @ corr_sub @ np.diag(vol)

n = len(saa_classes)
print(f"\n▸ SAA universe: {n} asset classes")

# ── 3. MVO optimization ───────────────────────────────────

# Objective: minimize (lambda/2) w'Sigma*w - mu'w
def objective(w):
    ret = mu @ w
    risk = w @ cov_matrix @ w
    return (risk_aversion / 2) * risk - ret

# Group indices
eq_idx = [i for i,c in enumerate(saa_classes) if c in ["US Eq","EU Eq","JP Eq","Asia ex-JP Eq"]]
fi_idx = [i for i,c in enumerate(saa_classes) if c in ["Global IG","Global HY","EM HC Debt","EM LC Debt"]]
alt_idx = [i for i,c in enumerate(saa_classes) if c in ["Private Markets","Hedge Funds"]]
cash_idx = [i for i,c in enumerate(saa_classes) if c == "Cash"]

bounds = []
for i, c in enumerate(saa_classes):
    if c == "Cash":
        bounds.append((min_cash, max_weight))
    else:
        bounds.append((min_weight, max_weight))

constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    {"type": "ineq", "fun": lambda w: sum(w[i] for i in eq_idx) - equity_range[0]},
    {"type": "ineq", "fun": lambda w: equity_range[1] - sum(w[i] for i in eq_idx)},
    {"type": "ineq", "fun": lambda w: sum(w[i] for i in fi_idx) - fi_range[0]},
    {"type": "ineq", "fun": lambda w: fi_range[1] - sum(w[i] for i in fi_idx)},
    {"type": "ineq", "fun": lambda w: sum(w[i] for i in alt_idx) - alts_range[0]},
    {"type": "ineq", "fun": lambda w: alts_range[1] - sum(w[i] for i in alt_idx)},
]

w0 = np.ones(n) / n

result = minimize(objective, w0, method="SLSQP", bounds=bounds, constraints=constraints,
                  options={"maxiter": 1000, "ftol": 1e-12})

if not result.success:
    print(f"\n⚠ Optimizer warning: {result.message}")
else:
    print("\n✓ Optimization converged")

w_opt = result.x

# ── 4. Results ─────────────────────────────────────────────

saa_result = pd.DataFrame({
    "Asset Class": saa_classes,
    "Weight": w_opt,
    "Fund": rep["Fund"].values,
    "Exp Return": mu,
    "Volatility": vol,
})
saa_result["Contribution"] = saa_result["Weight"] * saa_result["Exp Return"]

port_return = mu @ w_opt
port_var = w_opt @ cov_matrix @ w_opt
port_vol = np.sqrt(port_var)
port_sharpe = (port_return - rf_rate) / port_vol

print("\n" + "="*70)
print("STRATEGIC ASSET ALLOCATION (SAA)")
print("="*70)
print(f"Risk aversion λ = {risk_aversion}")
print(f"Equity range: {equity_range}, FI range: {fi_range}, Alts range: {alts_range}")
print("-"*70)

for _, row in saa_result.iterrows():
    bar = "█" * int(row["Weight"] * 50)
    print(f"  {row['Asset Class']:20s}  {row['Weight']:6.1%}  {bar}")

print("-"*70)
print(f"  Portfolio Expected Return:  {port_return:.2%}")
print(f"  Portfolio Volatility:       {port_vol:.2%}")
print(f"  Portfolio Sharpe Ratio:     {port_sharpe:.3f}")
print(f"  Expected Annual Income:     ${port_return * investable_aum:,.0f}")
print(f"  Income Target:              ${income_target:,}")
print(f"  Income gap:                 ${port_return * investable_aum - income_target:+,.0f}")
print("="*70)

# Group summary
eq_w = sum(w_opt[i] for i in eq_idx)
fi_w = sum(w_opt[i] for i in fi_idx)
alt_w = sum(w_opt[i] for i in alt_idx)
cash_w = sum(w_opt[i] for i in cash_idx)
print(f"\n  Equity: {eq_w:.1%} | Fixed Income: {fi_w:.1%} | Alternatives: {alt_w:.1%} | Cash: {cash_w:.1%}")

▸ Representative funds per asset class (Sharpe #1)
                                                 Fund  Return  Volatility    Sharpe
Asia ex-JP Eq       Templeton Asian Smaller Companies   0.088       0.155  0.341935
EM HC Debt            BlackRock Emerging Markets Bond   0.032       0.098 -0.030612
EM LC Debt                       BlackRock China Bond   0.014       0.107 -0.196262
EU Eq                  HSBC GIF Euroland Value Equity   0.165       0.167  0.778443
Global HY                        AB Global High Yield   0.039       0.073  0.054795
Global IG           HSBC Global IG Securitised Credit   0.035       0.020  0.000000
JP Eq            Pictet Japanese Equity Opportunities   0.180       0.124  1.169355
US Eq                    JPMorgan America Equity Fund   0.144       0.156  0.698718
Private Markets  US Core Plus Real Estate Income Fund   0.089       0.113  0.477876
Hedge Funds                           Asia Equity L/S   0.196       0.107  1.504673
Cash                     

# Phase B

In [10]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Phase B — Current Portfolio Analysis
# ═══════════════════════════════════════════════════════════════

total_investment = 45_000_000
current_aum = 45_000_000

# ── Asset class average return & vol (from Appendix I funds) ──
class_stats = (funds[funds["Asset Class"] != "Thematic"]
               .groupby("Asset Class")
               .agg({"Return": "mean", "Volatility": "mean"})
               .rename(index={
                   "US Equities":"US Eq", "European Equities":"EU Eq",
                   "Japanese Equities":"JP Eq", "Asia ex-Japan Equities":"Asia ex-JP Eq",
                   "Global IG Bonds":"Global IG", "Global HY Bonds":"Global HY",
                   "EM Hard Currency Debt":"EM HC Debt", "EM Local Currency Debt":"EM LC Debt"
               }))
class_stats.loc["Cash"] = [rf_rate, 0.001]

# ── Current portfolio weights & classes ──
cp_classes = current_portfolio["Asset Class"].values
cp_weights = current_portfolio["Weight"].values

cp_ret = np.array([class_stats.loc[c, "Return"] for c in cp_classes])
cp_vol = np.array([class_stats.loc[c, "Volatility"] for c in cp_classes])

# ── Portfolio return (weighted average) ──
port_return = cp_weights @ cp_ret

# ── Portfolio volatility (using correlation matrix) ──
corr_cp = corr_final.loc[cp_classes, cp_classes].values
cov_cp = np.diag(cp_vol) @ corr_cp @ np.diag(cp_vol)
port_vol = np.sqrt(cp_weights @ cov_cp @ cp_weights)

# ── Sharpe ──
port_sharpe = (port_return - rf_rate) / port_vol

# ── Income estimate (yield-based) ──
yields = yield_assumptions.set_index("Asset Class")
cp_yield = np.array([yields.loc[c, "Est Yield"] for c in cp_classes])
port_yield = cp_weights @ cp_yield
port_capgain = port_return - port_yield
annual_income = port_yield * current_aum
annual_capgain = port_capgain * current_aum
annual_total = port_return * current_aum

# ── Output ──
print("="*60)
print("CURRENT PORTFOLIO ANALYSIS")
print("="*60)

for i, c in enumerate(cp_classes):
    print(f"  {c:16s}  w={cp_weights[i]:5.0%}  ret={cp_ret[i]:.1%}  vol={cp_vol[i]:.1%}")

print("-"*60)
print(f"  Portfolio Return:      {port_return:.2%}")
print(f"  Portfolio Volatility:  {port_vol:.2%}")
print(f"  Portfolio Sharpe:      {port_sharpe:.3f}")
print("-"*60)
print(f"  Income:                ${annual_income:,.0f}  ({port_yield:.2%})")
print(f"  Capital Gain:          ${annual_capgain:,.0f}  ({port_capgain:.2%})")
print(f"  Total:                 ${annual_total:,.0f}  ({port_return:.2%})")
print("-"*60)
print(f"  Income Target:         $300,000")
print(f"  Income Gap:            ${annual_income - 300_000:+,.0f}")
print("="*60)

CURRENT PORTFOLIO ANALYSIS
  US Eq             w=  70%  ret=11.3%  vol=17.4%
  Asia ex-JP Eq     w=   8%  ret=6.7%  vol=15.7%
  Global HY         w=   2%  ret=3.4%  vol=6.9%
  Cash              w=  20%  ret=3.5%  vol=0.1%
------------------------------------------------------------
  Portfolio Return:      9.19%
  Portfolio Volatility:  13.01%
  Portfolio Sharpe:      0.437
------------------------------------------------------------
  Income:                $621,000  (1.38%)
  Capital Gain:          $3,513,195  (7.81%)
  Total:                 $4,134,195  (9.19%)
------------------------------------------------------------
  Income Target:         $300,000
  Income Gap:            $+321,000


In [11]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: SAA sweep — parameter ranges
# ═══════════════════════════════════════════════════════════════

saa_ranges = {
    "US Eq":         [0.15, 0.20],
    "EU Eq":         [0.05, 0.05],
    "JP Eq":         [0.05, 0.10],
    "Asia ex-JP Eq": [0.10, 0.15],
    "Global IG":     [0.20, 0.25],
    "Global HY":     [0.05, 0.10],
    "EM HC Debt":    [0.05, 0.10],
    "EM LC Debt":    [0.00, 0.10],
    "Private Markets":[0.05, 0.10],
    "Hedge Funds":   [0.10, 0.15, 0.20],
    "Cash":          [0.05, 0.10],
}

cash_return_override = rf_rate


In [12]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: SAA sweep — gross portfolio metrics ($80M basis)
# ═══════════════════════════════════════════════════════════════

from itertools import product

class_map = {
    "US Equities":"US Eq","European Equities":"EU Eq",
    "Japanese Equities":"JP Eq","Asia ex-Japan Equities":"Asia ex-JP Eq",
    "Global IG Bonds":"Global IG","Global HY Bonds":"Global HY",
    "EM Hard Currency Debt":"EM HC Debt","EM Local Currency Debt":"EM LC Debt",
}

rep_trad = (funds[funds["Asset Class"] != "Thematic"]
            .groupby("Asset Class")
            .apply(lambda x: x.loc[x["Sharpe"].idxmax()])
            [["Fund","Return","Volatility","Sharpe"]])
rep_trad.index = rep_trad.index.map(class_map)

best_pm = alts[alts["Type"]=="Private Markets"].sort_values("Sharpe",ascending=False).iloc[0]
best_hf = alts[alts["Type"]=="Hedge Fund"].sort_values("Sharpe",ascending=False).iloc[0]
rep_alts = pd.DataFrame({
    "Fund":[best_pm["Fund"],best_hf["Fund"]],
    "Return":[best_pm["Return"],best_hf["Return"]],
    "Volatility":[best_pm["Volatility"],best_hf["Volatility"]],
    "Sharpe":[best_pm["Sharpe"],best_hf["Sharpe"]],
}, index=["Private Markets","Hedge Funds"])
rep_cash = pd.DataFrame({
    "Fund":["Cash"],"Return":[cash_return_override],"Volatility":[0.001],"Sharpe":[0.0]
}, index=["Cash"])
rep = pd.concat([rep_trad, rep_alts, rep_cash])

saa_classes = list(saa_ranges.keys())
mu = np.array([rep.loc[c,"Return"] for c in saa_classes])
vol_vec = np.array([rep.loc[c,"Volatility"] for c in saa_classes])
corr_sub = corr_final.loc[saa_classes, saa_classes].values
cov_matrix = np.diag(vol_vec) @ corr_sub @ np.diag(vol_vec)

yields_df = yield_assumptions.set_index("Asset Class")
yield_vec = np.array([yields_df.loc[c,"Est Yield"] for c in saa_classes])

all_vals = [saa_ranges[c] for c in saa_classes]
total = 1
for v in all_vals:
    total *= len(v)
print(f"Total combinations: {total}")

results = []
for combo in product(*all_vals):
    w = np.array(combo)
    if abs(w.sum() - 1.0) > 0.001:
        continue

    p_ret = mu @ w
    p_vol = np.sqrt(w @ cov_matrix @ w)
    p_sharpe = (p_ret - rf_rate) / p_vol if p_vol > 0.001 else 0.0
    p_yield = yield_vec @ w
    p_capgain = p_ret - p_yield

    results.append({
        **{c: w[i] for i, c in enumerate(saa_classes)},
        "Return": p_ret,
        "Vol": p_vol,
        "Sharpe": p_sharpe,
        "Cash Ret": p_yield,
        "Cap Gain": p_capgain,
    })

df = pd.DataFrame(results)
df = df.sort_values("Sharpe", ascending=False)
print(f"Valid (sum=100%): {len(df)}")

# ── Top 10 ──
print("\n" + "="*80)
print(f"TOP 10 BY SHARPE (on ${total_investment/1e6:.0f}M)")
print("="*80)
for rank, (_, row) in enumerate(df.head(10).iterrows(), 1):
    weights = " | ".join(f"{c}:{row[c]:.0%}" for c in saa_classes if row[c] > 0.005)
    total_dollar = row["Return"] * total_investment
    income_dollar = row["Cash Ret"] * total_investment
    capgain_dollar = row["Cap Gain"] * total_investment
    print(f"\n  #{rank}  Sharpe={row['Sharpe']:.3f}  Return={row['Return']:.1%}  Vol={row['Vol']:.1%}")
    print(f"       Total:    ${total_dollar:,.0f}  ({row['Return']:.2%})")
    print(f"       Income:   ${income_dollar:,.0f}  ({row['Cash Ret']:.2%})")
    print(f"       Cap Gain: ${capgain_dollar:,.0f}  ({row['Cap Gain']:.2%})")
    print(f"       {weights}")

# ── Summary ──
print("\n" + "="*80)
print(f"SUMMARY (on ${total_investment/1e6:.0f}M)")
print("="*80)
print(f"  Return range:    {df['Return'].min():.1%} ~ {df['Return'].max():.1%}")
print(f"  Vol range:       {df['Vol'].min():.1%} ~ {df['Vol'].max():.1%}")
print(f"  Sharpe range:    {df['Sharpe'].min():.3f} ~ {df['Sharpe'].max():.3f}")
print(f"  Total $ range:   ${df['Return'].min()*total_investment:,.0f} ~ ${df['Return'].max()*total_investment:,.0f}")
print(f"  Income $ range:  ${df['Cash Ret'].min()*total_investment:,.0f} ~ ${df['Cash Ret'].max()*total_investment:,.0f}")

Total combinations: 3072
Valid (sum=100%): 202

TOP 10 BY SHARPE (on $45M)

  #1  Sharpe=1.030  Return=11.3%  Vol=7.5%
       Total:    $5,067,000  (11.26%)
       Income:   $1,203,750  (2.68%)
       Cap Gain: $3,863,250  (8.59%)
       US Eq:15% | EU Eq:5% | JP Eq:10% | Asia ex-JP Eq:10% | Global IG:20% | Global HY:5% | EM HC Debt:5% | Private Markets:5% | Hedge Funds:20% | Cash:5%

  #2  Sharpe=1.030  Return=11.3%  Vol=7.5%
       Total:    $5,067,000  (11.26%)
       Income:   $1,203,750  (2.68%)
       Cap Gain: $3,863,250  (8.59%)
       US Eq:15% | EU Eq:5% | JP Eq:10% | Asia ex-JP Eq:10% | Global IG:20% | Global HY:5% | EM HC Debt:5% | Private Markets:5% | Hedge Funds:20% | Cash:5%

  #3  Sharpe=1.004  Return=10.5%  Vol=7.0%
       Total:    $4,740,750  (10.54%)
       Income:   $1,237,500  (2.75%)
       Cap Gain: $3,503,250  (7.79%)
       US Eq:15% | EU Eq:5% | JP Eq:5% | Asia ex-JP Eq:10% | Global IG:20% | Global HY:5% | EM HC Debt:5% | Private Markets:5% | Hedge Funds:20% 

In [13]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: Key scenarios for comparison table
# ═══════════════════════════════════════════════════════════════

# ── Pick scenarios from sweep results ──

recommended = df.iloc[0]  # #1 Sharpe
conservative = df.loc[df["Vol"].idxmin()]  # lowest vol
max_income = df.loc[df["Cash Ret"].idxmax()]  # highest yield

scenarios = {
    "Current Portfolio": {
        "Return": port_return, "Vol": port_vol, "Sharpe": port_sharpe,
        "Cash Ret": cp_weights @ cp_yield, "Cap Gain": port_return - (cp_weights @ cp_yield),
    },
    "Conservative": {c: conservative[c] for c in ["Return","Vol","Sharpe","Cash Ret","Cap Gain"]},
    "Recommended": {c: recommended[c] for c in ["Return","Vol","Sharpe","Cash Ret","Cap Gain"]},
    "Max Income": {c: max_income[c] for c in ["Return","Vol","Sharpe","Cash Ret","Cap Gain"]},
}

print("="*80)
print(f"SCENARIO COMPARISON (on ${total_investment/1e6:.0f}M)")
print("="*80)
print(f"  {'Scenario':20s} {'Return':>8s} {'Vol':>8s} {'Sharpe':>8s} {'Income':>12s} {'Cap Gain':>12s} {'Total':>12s}")
print("-"*80)

for name, s in scenarios.items():
    income_d = s["Cash Ret"] * total_investment
    capgain_d = s["Cap Gain"] * total_investment
    total_d = s["Return"] * total_investment
    print(f"  {name:20s} {s['Return']:7.1%} {s['Vol']:7.1%} {s['Sharpe']:8.3f} ${income_d:>10,.0f} ${capgain_d:>10,.0f} ${total_d:>10,.0f}")

# ── Show weights for each proposed scenario ──
print("\n" + "="*80)
print("WEIGHT BREAKDOWN")
print("="*80)

for label, row in [("Conservative", conservative), ("Recommended", recommended), ("Max Income", max_income)]:
    weights = " | ".join(f"{c}:{row[c]:.0%}" for c in saa_classes if row[c] > 0.005)
    print(f"\n  {label}:")
    print(f"    {weights}")

SCENARIO COMPARISON (on $45M)
  Scenario               Return      Vol   Sharpe       Income     Cap Gain        Total
--------------------------------------------------------------------------------
  Current Portfolio       9.2%   13.0%    0.437 $   621,000 $ 3,513,195 $ 4,134,195
  Conservative            9.7%    6.8%    0.915 $ 1,305,000 $ 3,073,500 $ 4,378,500
  Recommended            11.3%    7.5%    1.030 $ 1,203,750 $ 3,863,250 $ 5,067,000
  Max Income              9.2%    7.5%    0.755 $ 1,485,000 $ 2,655,000 $ 4,140,000

WEIGHT BREAKDOWN

  Conservative:
    US Eq:15% | EU Eq:5% | JP Eq:5% | Asia ex-JP Eq:10% | Global IG:25% | Global HY:5% | EM HC Debt:5% | Private Markets:5% | Hedge Funds:15% | Cash:10%

  Recommended:
    US Eq:15% | EU Eq:5% | JP Eq:10% | Asia ex-JP Eq:10% | Global IG:20% | Global HY:5% | EM HC Debt:5% | Private Markets:5% | Hedge Funds:20% | Cash:5%

  Max Income:
    US Eq:15% | EU Eq:5% | JP Eq:5% | Asia ex-JP Eq:10% | Global IG:20% | Global HY:10% | EM

In [14]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: Income & education — parameters
# ═══════════════════════════════════════════════════════════════

# Income
monthly_expense = 50_000
income_target_annual = monthly_expense * 0.5 * 12  # $300K

# Education
children = 2
current_age = 5
university_age = 18
years_until_uni = university_age - current_age  # 13 years
university_years = 4
annual_uni_cost_today = 85_000  # USD, current value (tuition + living)
education_inflation = 0.04      # annual education cost inflation
discount_rate = rf_rate

In [15]:
# ═══════════════════════════════════════════════════════════════
# CELL 12: Income & education — calculations
# ═══════════════════════════════════════════════════════════════

# ── 1. Income sufficiency ──────────────────────────────────

portfolio_income = recommended["Cash Ret"] * total_investment
income_surplus = portfolio_income - income_target_annual

print("="*60)
print("INCOME SUFFICIENCY")
print("="*60)
print(f"  Portfolio yield:     {recommended['Cash Ret']:.2%}")
print(f"  Annual income:       ${portfolio_income:,.0f}")
print(f"  Target:              ${income_target_annual:,.0f}")
print(f"  Surplus:             ${income_surplus:+,.0f}")
print(f"  Coverage ratio:      {portfolio_income/income_target_annual:.1f}x")

# ── 2. Education cost projection ───────────────────────────

# Future cost per year (inflated)
edu_costs_per_child = []
for y in range(university_years):
    year_from_now = years_until_uni + y
    future_cost = annual_uni_cost_today * (1 + education_inflation) ** year_from_now
    edu_costs_per_child.append(future_cost)

total_per_child = sum(edu_costs_per_child)
total_both = total_per_child * children

# Present value
pv_per_child = sum(
    annual_uni_cost_today * (1 + education_inflation) ** (years_until_uni + y)
    / (1 + discount_rate) ** (years_until_uni + y)
    for y in range(university_years)
)
pv_both = pv_per_child * children

print("\n" + "="*60)
print("EDUCATION COST PROJECTION")
print("="*60)
print(f"  Children:            {children} (twins, age {current_age})")
print(f"  University in:       {years_until_uni} years")
print(f"  Duration:            {university_years} years each")
print(f"  Annual cost today:   ${annual_uni_cost_today:,.0f}")
print(f"  Education inflation: {education_inflation:.0%}")
print(f"  Discount rate:       {discount_rate:.2%} (portfolio return)")

print(f"\n  Future cost per child:")
for y, cost in enumerate(edu_costs_per_child):
    print(f"    Year {years_until_uni+y}: ${cost:,.0f}")
print(f"  Total per child:     ${total_per_child:,.0f}")
print(f"  Total both:          ${total_both:,.0f}")

print(f"\n  Present value:")
print(f"    Per child:         ${pv_per_child:,.0f}")
print(f"    Both:              ${pv_both:,.0f}")
print(f"    As % of AUM:       {pv_both/total_investment:.1%}")

# ── 3. Summary ─────────────────────────────────────────────

print("\n" + "="*60)
print("FINANCIAL GOALS SUMMARY")
print("="*60)
print(f"  Lifestyle income:    ${income_target_annual:,.0f}/yr  → {'✓ MET' if income_surplus >= 0 else '✗ NOT MET'} (surplus ${income_surplus:+,.0f})")
print(f"  Education reserve:   ${pv_both:,.0f} needed today")
print(f"  Remaining AUM:       ${total_investment - pv_both:,.0f} ({(total_investment-pv_both)/total_investment:.1%})")

INCOME SUFFICIENCY
  Portfolio yield:     2.68%
  Annual income:       $1,203,750
  Target:              $300,000
  Surplus:             $+903,750
  Coverage ratio:      4.0x

EDUCATION COST PROJECTION
  Children:            2 (twins, age 5)
  University in:       13 years
  Duration:            4 years each
  Annual cost today:   $85,000
  Education inflation: 4%
  Discount rate:       3.50% (portfolio return)

  Future cost per child:
    Year 13: $141,531
    Year 14: $147,192
    Year 15: $153,080
    Year 16: $159,203
  Total per child:     $601,007
  Total both:          $1,202,015

  Present value:
    Per child:         $364,614
    Both:              $729,228
    As % of AUM:       1.6%

FINANCIAL GOALS SUMMARY
  Lifestyle income:    $300,000/yr  → ✓ MET (surplus $+903,750)
  Education reserve:   $729,228 needed today
  Remaining AUM:       $44,270,772 (98.4%)


In [16]:
# ═══════════════════════════════════════════════════════════════
# CELL: TAA — Tactical Asset Allocation Metrics
# ═══════════════════════════════════════════════════════════════
# Dependencies: Cell 0-6 (data, rf_rate, corr_final, yield_assumptions)
#               Cell 8-9 (rep, saa_classes, mu, vol_vec, cov_matrix, yield_vec)
#               Cell 10  (recommended = SAA baseline)
# ═══════════════════════════════════════════════════════════════

# ── 1. Define SAA and TAA weights ─────────────────────────────

taa_weights = {
    "US Eq":           0.12,
    "EU Eq":           0.06,
    "JP Eq":           0.10,
    "Asia ex-JP Eq":   0.07,
    "Global IG":       0.25,
    "Global HY":       0.05,
    "EM HC Debt":      0.05,
    "EM LC Debt":      0.00,
    "Private Markets": 0.05,
    "Hedge Funds":     0.20,
    "Cash":            0.05,
}

saa_weights = {
    "US Eq":           0.15,
    "EU Eq":           0.05,
    "JP Eq":           0.10,
    "Asia ex-JP Eq":   0.10,
    "Global IG":       0.20,
    "Global HY":       0.05,
    "EM HC Debt":      0.05,
    "EM LC Debt":      0.00,
    "Private Markets": 0.05,
    "Hedge Funds":     0.20,
    "Cash":            0.05,
}

w_saa = np.array([saa_weights[c] for c in saa_classes])
w_taa = np.array([taa_weights[c] for c in saa_classes])

assert abs(w_taa.sum() - 1.0) < 0.001, f"TAA weights sum to {w_taa.sum():.3f}, not 1.0"

# ── 2. Portfolio metrics ──────────────────────────────────────

# SAA (recalculate for clean comparison)
saa_ret    = mu @ w_saa
saa_vol    = np.sqrt(w_saa @ cov_matrix @ w_saa)
saa_sharpe = (saa_ret - rf_rate) / saa_vol
saa_yield  = yield_vec @ w_saa
saa_capgain = saa_ret - saa_yield

# TAA
taa_ret    = mu @ w_taa
taa_vol    = np.sqrt(w_taa @ cov_matrix @ w_taa)
taa_sharpe = (taa_ret - rf_rate) / taa_vol
taa_yield  = yield_vec @ w_taa
taa_capgain = taa_ret - taa_yield

# Weighted AR
ar_map = {}
for c in saa_classes:
    if c == "Cash":
        ar_map[c] = 0.0
    elif c in ["Private Markets", "Hedge Funds"]:
        best = alts[alts["Type"] == ("Private Markets" if c == "Private Markets" else "Hedge Fund")].sort_values("Sharpe", ascending=False).iloc[0]
        ar_map[c] = best["AR"]
    else:
        inv_class_map = {"US Eq":"US Equities","EU Eq":"European Equities","JP Eq":"Japanese Equities",
                         "Asia ex-JP Eq":"Asia ex-Japan Equities","Global IG":"Global IG Bonds",
                         "Global HY":"Global HY Bonds","EM HC Debt":"EM Hard Currency Debt",
                         "EM LC Debt":"EM Local Currency Debt"}
        full_name = inv_class_map.get(c, c)
        best_fund = funds[funds["Asset Class"] == full_name].sort_values("Sharpe", ascending=False).iloc[0]
        ar_map[c] = best_fund["AR"]

ar_vec = np.array([ar_map[c] for c in saa_classes])
saa_war = w_saa @ ar_vec
taa_war = w_taa @ ar_vec

# ── 3. Output ─────────────────────────────────────────────────

print("=" * 70)
print("SAA vs TAA COMPARISON")
print("=" * 70)

# Header
print(f"\n  {'Asset Class':18s} {'SAA':>6s} {'TAA':>6s} {'Tilt':>7s}  {'AR':>5s}")
print("-" * 50)

eq_classes = ["US Eq", "EU Eq", "JP Eq", "Asia ex-JP Eq"]
fi_classes = ["Global IG", "Global HY", "EM HC Debt", "EM LC Debt"]
alt_classes = ["Private Markets", "Hedge Funds"]
cash_classes = ["Cash"]

def print_group(label, classes, saa_w, taa_w):
    saa_sum = sum(saa_w[saa_classes.index(c)] for c in classes)
    taa_sum = sum(taa_w[saa_classes.index(c)] for c in classes)
    for c in classes:
        i = saa_classes.index(c)
        s, t = saa_w[i], taa_w[i]
        tilt = t - s
        ar = ar_map[c]
        tilt_str = f"{tilt:+.0%}" if abs(tilt) > 0.001 else "—"
        print(f"  {c:18s} {s:5.0%}  {t:5.0%}  {tilt_str:>6s}  {ar:4.0%}")
    print(f"  {'Total '+label:18s} {saa_sum:5.0%}  {taa_sum:5.0%}  {taa_sum-saa_sum:+.0%}")
    print()

print_group("Equity", eq_classes, w_saa, w_taa)
print_group("FI", fi_classes, w_saa, w_taa)
print_group("Alts", alt_classes, w_saa, w_taa)
print_group("Cash", cash_classes, w_saa, w_taa)

# Metrics comparison
print("=" * 70)
print(f"  {'Metric':25s} {'SAA':>12s} {'TAA':>12s} {'Delta':>10s}")
print("-" * 70)
print(f"  {'Expected Return':25s} {saa_ret:11.2%}  {taa_ret:11.2%}  {taa_ret-saa_ret:+9.2%}")
print(f"  {'Expected Volatility':25s} {saa_vol:11.2%}  {taa_vol:11.2%}  {taa_vol-saa_vol:+9.2%}")
print(f"  {'Sharpe Ratio':25s} {saa_sharpe:11.3f}  {taa_sharpe:11.3f}  {taa_sharpe-saa_sharpe:+9.3f}")
print(f"  {'Weighted AR':25s} {saa_war:11.1%}  {taa_war:11.1%}  {taa_war-saa_war:+9.1%}")
print("-" * 70)
print(f"  {'Cash Return (Yield)':25s} {saa_yield:11.2%}  {taa_yield:11.2%}  {taa_yield-saa_yield:+9.2%}")
print(f"  {'Capital Gain':25s} {saa_capgain:11.2%}  {taa_capgain:11.2%}  {taa_capgain-saa_capgain:+9.2%}")
print("=" * 70)

# Dollar terms
print(f"\n  On ${total_investment/1e6:.0f}M:")
print(f"  {'':25s} {'SAA':>12s} {'TAA':>12s}")
print(f"  {'Total Return':25s} ${saa_ret*total_investment:>11,.0f} ${taa_ret*total_investment:>11,.0f}")
print(f"  {'Income (Yield)':25s} ${saa_yield*total_investment:>11,.0f} ${taa_yield*total_investment:>11,.0f}")
print(f"  {'Capital Gain':25s} ${saa_capgain*total_investment:>11,.0f} ${taa_capgain*total_investment:>11,.0f}")
print(f"  {'Borrowing Capacity (WAR)':25s} ${saa_war*total_investment:>11,.0f} ${taa_war*total_investment:>11,.0f}")

SAA vs TAA COMPARISON

  Asset Class           SAA    TAA    Tilt     AR
--------------------------------------------------
  US Eq                15%    12%     -3%   70%
  EU Eq                 5%     6%     +1%   70%
  JP Eq                10%    10%       —   70%
  Asia ex-JP Eq        10%     7%     -3%   70%
  Total Equity         40%    35%  -5%

  Global IG            20%    25%     +5%   80%
  Global HY             5%     5%       —   60%
  EM HC Debt            5%     5%       —   75%
  EM LC Debt            0%     0%       —   75%
  Total FI             30%    35%  +5%

  Private Markets       5%     5%       —   50%
  Hedge Funds          20%    20%       —   40%
  Total Alts           25%    25%  +0%

  Cash                  5%     5%       —    0%
  Total Cash            5%     5%  +0%

  Metric                             SAA          TAA      Delta
----------------------------------------------------------------------
  Expected Return                11.26%       10.90%

In [17]:
# ═══════════════════════════════════════════════════════════════
# CELL 13a: Credit Solution — Parameters
# ═══════════════════════════════════════════════════════════════
# Dependencies: Cell 0 (external_assets), Cell 1 (borrowing_cost)
# Modify flags below based on team decisions, then run Cell 13b
# ═══════════════════════════════════════════════════════════════

# ── TEAM DECISIONS (Q1-Q3) ────────────────────────────────────

# Q1: Which external assets to pledge as collateral?
use_liondash = True       # $30M, AR 50% — CEO stake, concentration risk
use_property = True       # $15M, AR 50% — Singapore commercial property
use_fine_art = True       # $5M,  AR 75% — valuation uncertainty

# Q2: What does "$80M investment" mean?
#   True  = $80M is the portfolio size (pre-AR). AR creates additional
#           borrowing capacity on top (for lifestyle, reinvestment, etc.)
#   False = $80M is gross exposure including AR leverage.
#           Client puts less equity, AR bridges the rest.
target_is_equity = False

# Q3: How to structure the credit facility?
#   True  = Multi-stage: external collateral + fund AR leveraging
#   False = Single-stage: external collateral only
multi_stage = True

# ── FIXED PARAMETERS ─────────────────────────────────────────

borrowing_cost = 0.05         # SOFR (~4.3%) + PB spread (~75bps)
investable_cash = 45_000_000
target_investment = 80_000_000
funding_gap = target_investment - investable_cash  # $35M

# ── EXTERNAL ASSET COLLATERAL ────────────────────────────────

ext_assets = pd.DataFrame({
    "Asset":    ["LionDash Shares", "Singapore Commercial Property", "Fine Art"],
    "Value":    [30_000_000, 15_000_000, 5_000_000],
    "AR":       [0.50, 0.50, 0.75],
    "Use":      [use_liondash, use_property, use_fine_art],
})
ext_assets["Borrowing Capacity"] = ext_assets["Value"] * ext_assets["AR"] * ext_assets["Use"]

# ── VALIDATION ───────────────────────────────────────────────

total_ext_capacity = ext_assets["Borrowing Capacity"].sum()
print("=" * 60)
print("CREDIT SOLUTION — PARAMETERS")
print("=" * 60)
print(f"  Investable cash:       ${investable_cash/1e6:.0f}M")
print(f"  Target investment:     ${target_investment/1e6:.0f}M")
print(f"  Funding gap:           ${funding_gap/1e6:.0f}M")
print(f"  Borrowing cost:        {borrowing_cost:.2%} (SOFR + 75bps)")
print()
print("  External collateral:")
for _, row in ext_assets.iterrows():
    status = "✓" if row["Use"] else "✗"
    cap = f"${row['Borrowing Capacity']/1e6:.2f}M" if row["Use"] else "—"
    print(f"    {status} {row['Asset']:35s} ${row['Value']/1e6:.0f}M × {row['AR']:.0%} = {cap}")
print(f"  Total external capacity:  ${total_ext_capacity/1e6:.2f}M")
print()
print(f"  Target interpretation:    {'$80M is portfolio equity (AR on top)' if target_is_equity else '$80M is gross exposure (AR included)'}")
print(f"  Structure:                {'Multi-stage (external + fund AR)' if multi_stage else 'Single-stage (external only)'}")

if total_ext_capacity < funding_gap and not multi_stage:
    print(f"\n  ⚠ External capacity ${total_ext_capacity/1e6:.2f}M < gap ${funding_gap/1e6:.0f}M")
    print(f"    → Need multi-stage or reduce target")

CREDIT SOLUTION — PARAMETERS
  Investable cash:       $45M
  Target investment:     $80M
  Funding gap:           $35M
  Borrowing cost:        5.00% (SOFR + 75bps)

  External collateral:
    ✓ LionDash Shares                     $30M × 50% = $15.00M
    ✓ Singapore Commercial Property       $15M × 50% = $7.50M
    ✓ Fine Art                            $5M × 75% = $3.75M
  Total external capacity:  $26.25M

  Target interpretation:    $80M is gross exposure (AR included)
  Structure:                Multi-stage (external + fund AR)


In [18]:
# ═══════════════════════════════════════════════════════════════
# CELL 13b: Credit Solution — Calculations
# ═══════════════════════════════════════════════════════════════
# Dependencies: Cell 13a (parameters), Cell 8-9 (SAA/TAA weights,
#               rep funds, ar_map), Cell 0 (all fund data)
# ═══════════════════════════════════════════════════════════════

# ── 1. Weighted AR of portfolio ──────────────────────────────

# Use TAA weights (final portfolio); falls back to SAA if taa_weights not defined
try:
    portfolio_weights = taa_weights
    portfolio_label = "TAA"
except NameError:
    portfolio_weights = saa_weights
    portfolio_label = "SAA"

w_portfolio = np.array([portfolio_weights[c] for c in saa_classes])
weighted_ar = w_portfolio @ ar_vec   # ar_vec from TAA cell

# ── 2. Funding structure ─────────────────────────────────────

if target_is_equity:
    # ═══ SCENARIO A: $80M is portfolio equity, AR creates capacity on top ═══
    
    # Step 1: Fund the $80M
    external_borrowing = min(total_ext_capacity, funding_gap)
    remaining_gap = funding_gap - external_borrowing
    
    if remaining_gap > 0 and not multi_stage:
        print(f"⚠ Cannot fund ${target_investment/1e6:.0f}M with single-stage.")
        print(f"   Gap remaining: ${remaining_gap/1e6:.2f}M")
    
    # If multi-stage and external isn't enough, use partial fund AR to bridge
    if remaining_gap > 0 and multi_stage:
        # Invest what we have first, then AR-borrow to cover remaining
        initial_investment = investable_cash + external_borrowing
        ar_borrowing_for_gap = remaining_gap  # borrow from fund AR to fill gap
    else:
        initial_investment = target_investment
        ar_borrowing_for_gap = 0
    
    # Step 2: AR capacity on the full $80M portfolio
    total_ar_capacity = weighted_ar * target_investment
    ar_used_for_gap = ar_borrowing_for_gap
    ar_available = total_ar_capacity - ar_used_for_gap  # remaining AR capacity
    
    # Total borrowing
    total_borrowing = external_borrowing + ar_borrowing_for_gap
    
    # Gross exposure if client uses remaining AR
    gross_exposure_max = target_investment + ar_available
    
else:
    # ═══ SCENARIO B: $80M is gross exposure, AR included ═══
    
    # equity_needed = $80M × (1 - WAR)
    equity_needed = target_investment * (1 - weighted_ar)
    external_borrowing = min(total_ext_capacity, max(0, equity_needed - investable_cash))
    
    total_equity = investable_cash + external_borrowing
    
    if total_equity < equity_needed:
        print(f"⚠ Equity shortfall: need ${equity_needed/1e6:.2f}M, have ${total_equity/1e6:.2f}M")
    
    ar_borrowing_for_gap = target_investment - total_equity
    total_borrowing = external_borrowing + ar_borrowing_for_gap
    ar_available = 0  # fully utilized
    gross_exposure_max = target_investment

# ── 3. Cost analysis ─────────────────────────────────────────

# Interest on external borrowing (Lombard facility)
annual_interest_external = external_borrowing * borrowing_cost

# Interest on fund AR borrowing
annual_interest_ar = ar_borrowing_for_gap * borrowing_cost

# Total interest
total_annual_interest = annual_interest_external + annual_interest_ar

# Portfolio return (gross, on $80M)
port_ret_pct = mu @ w_portfolio
gross_return = port_ret_pct * target_investment

# Net return
net_return = gross_return - total_annual_interest
net_return_on_equity = net_return / investable_cash  # return on client's own money

# Income (yield) vs interest
port_yield_pct = yield_vec @ w_portfolio
gross_income = port_yield_pct * target_investment
net_income = gross_income - total_annual_interest

# ── 4. Per-asset-class AR breakdown ──────────────────────────

ar_breakdown = pd.DataFrame({
    "Asset Class": saa_classes,
    "Weight": w_portfolio,
    "Allocation ($M)": w_portfolio * target_investment / 1e6,
    "Fund AR": ar_vec,
    "AR Capacity ($M)": w_portfolio * target_investment * ar_vec / 1e6,
})
ar_breakdown = ar_breakdown[ar_breakdown["Weight"] > 0.001]

# ── 5. Output ────────────────────────────────────────────────

print("=" * 70)
print("CREDIT SOLUTION")
print("=" * 70)

# ── Funding structure ──
print(f"\n{'─'*70}")
print("FUNDING STRUCTURE")
print(f"{'─'*70}")
print(f"  Client cash:                      ${investable_cash/1e6:.0f}M")
print(f"  External asset Lombard:           ${external_borrowing/1e6:.2f}M")
if ar_borrowing_for_gap > 0:
    print(f"  Fund AR borrowing (gap fill):     ${ar_borrowing_for_gap/1e6:.2f}M")
print(f"  {'─'*40}")
print(f"  Total invested (portfolio):       ${target_investment/1e6:.0f}M")
print(f"  Total borrowing:                  ${total_borrowing/1e6:.2f}M")
print(f"  Leverage ratio:                   {target_investment/investable_cash:.2f}x (on client equity)")

# ── External collateral detail ──
print(f"\n{'─'*70}")
print("EXTERNAL COLLATERAL")
print(f"{'─'*70}")
for _, row in ext_assets.iterrows():
    if row["Use"]:
        print(f"  {row['Asset']:35s}  ${row['Value']/1e6:.0f}M × {row['AR']:.0%} = ${row['Borrowing Capacity']/1e6:.2f}M")
print(f"  {'─'*40}")
print(f"  Total Lombard capacity:           ${total_ext_capacity/1e6:.2f}M")
print(f"  Lombard utilized:                 ${external_borrowing/1e6:.2f}M")

# ── Fund AR breakdown ──
print(f"\n{'─'*70}")
print(f"FUND AR BREAKDOWN ({portfolio_label} weights)")
print(f"{'─'*70}")
print(f"  {'Asset Class':18s} {'Weight':>7s} {'Alloc($M)':>10s} {'AR':>5s} {'AR Cap($M)':>11s}")
print(f"  {'─'*55}")
for _, row in ar_breakdown.iterrows():
    print(f"  {row['Asset Class']:18s} {row['Weight']:6.0%}  ${row['Allocation ($M)']:8.1f}M  {row['Fund AR']:4.0%}  ${row['AR Capacity ($M)']:8.1f}M")
print(f"  {'─'*55}")
print(f"  {'Total':18s} {w_portfolio.sum():6.0%}  ${target_investment/1e6:8.1f}M  {weighted_ar:4.0%}  ${weighted_ar*target_investment/1e6:8.1f}M")

if target_is_equity:
    print(f"\n  AR used for gap fill:             ${ar_used_for_gap/1e6:.2f}M")
    print(f"  AR remaining (available credit):   ${ar_available/1e6:.2f}M")
    print(f"  Max gross exposure:               ${gross_exposure_max/1e6:.2f}M")

# ── Cost / Return analysis ──
print(f"\n{'─'*70}")
print("RETURN AFTER BORROWING COSTS")
print(f"{'─'*70}")
print(f"  Gross portfolio return:           {port_ret_pct:.2%}  =  ${gross_return/1e6:.2f}M")
print(f"  Gross income (yield):             {port_yield_pct:.2%}  =  ${gross_income/1e6:.2f}M")
print(f"  {'─'*40}")
print(f"  Interest — Lombard facility:      ${annual_interest_external/1e6:.2f}M  (${external_borrowing/1e6:.1f}M × {borrowing_cost:.1%})")
if ar_borrowing_for_gap > 0:
    print(f"  Interest — Fund AR gap fill:      ${annual_interest_ar/1e6:.2f}M  (${ar_borrowing_for_gap/1e6:.1f}M × {borrowing_cost:.1%})")
print(f"  Total annual interest:            ${total_annual_interest/1e6:.2f}M")
print(f"  {'─'*40}")
print(f"  Net return:                       ${net_return/1e6:.2f}M")
print(f"  Net return on client equity:      {net_return_on_equity:.2%}  (on ${investable_cash/1e6:.0f}M)")
print(f"  Net income after interest:        ${net_income/1e6:.2f}M")
print(f"  Income target ($300K):            {'✓ MET' if net_income >= 300_000 else '✗ NOT MET'}  ({net_income/300_000:.1f}x coverage)")

# ── Summary for slides ──
print(f"\n{'='*70}")
print("SLIDE SUMMARY")
print(f"{'='*70}")
print(f"  Portfolio size:     ${target_investment/1e6:.0f}M")
print(f"  Client equity:      ${investable_cash/1e6:.0f}M")
print(f"  Lombard borrowing:  ${external_borrowing/1e6:.1f}M (external collateral)")
if ar_borrowing_for_gap > 0:
    print(f"  AR borrowing:       ${ar_borrowing_for_gap/1e6:.1f}M (fund AR)")
print(f"  Borrowing cost:     {borrowing_cost:.1%} (SOFR + 75bps)")
print(f"  Gross return:       {port_ret_pct:.2%}  →  Net: {net_return/target_investment:.2%}")
print(f"  Weighted AR:        {weighted_ar:.1%}")
if target_is_equity:
    print(f"  AR credit line:     ${ar_available/1e6:.1f}M (available for lifestyle/reinvestment)")
print(f"  Return on equity:   {net_return_on_equity:.2%}")

CREDIT SOLUTION

──────────────────────────────────────────────────────────────────────
FUNDING STRUCTURE
──────────────────────────────────────────────────────────────────────
  Client cash:                      $45M
  External asset Lombard:           $0.00M
  Fund AR borrowing (gap fill):     $35.00M
  ────────────────────────────────────────
  Total invested (portfolio):       $80M
  Total borrowing:                  $35.00M
  Leverage ratio:                   1.78x (on client equity)

──────────────────────────────────────────────────────────────────────
EXTERNAL COLLATERAL
──────────────────────────────────────────────────────────────────────
  LionDash Shares                      $30M × 50% = $15.00M
  Singapore Commercial Property        $15M × 50% = $7.50M
  Fine Art                             $5M × 75% = $3.75M
  ────────────────────────────────────────
  Total Lombard capacity:           $26.25M
  Lombard utilized:                 $0.00M

───────────────────────────────────

In [21]:
# ═══════════════════════════════════════════════════════════════
# CELL: Fund Selection — Equal vs Sharpe-Proportional
# ═══════════════════════════════════════════════════════════════
# Dependencies: Cell 0-3 (funds, alts, Sharpe), SAA cell
# ═══════════════════════════════════════════════════════════════

total_investment = 45_000_000

selected = [
    ("US Eq", 0.15, "AB American Growth Portfolio"),
    ("US Eq", 0.15, "JPMorgan America Equity Fund"),
    ("EU Eq", 0.05, "HSBC GIF Euroland Value Equity"),
    ("EU Eq", 0.05, "JPMorgan Europe Equity Plus"),
    ("EU Eq", 0.05, "Fidelity European Growth Fund"),
    ("JP Eq", 0.10, "M&G Japan"),
    ("JP Eq", 0.10, "Pictet Japanese Equity Opportunities"),
    ("Asia ex-JP Eq", 0.10, "Schroder Asian Equity Yield"),
    ("Asia ex-JP Eq", 0.10, "Templeton Asian Smaller Companies"),
    ("Global IG", 0.20, "HSBC Global IG Securitised Credit"),
    ("Global IG", 0.20, "PIMCO Income"),
    ("Global IG", 0.20, "Schroder Global Credit Income"),
    ("Global HY", 0.05, "BlackRock Global High Yield Bond"),
    ("Global HY", 0.05, "Schroder Global High Yield"),
    ("EM HC Debt", 0.05, "BlackRock Emerging Markets Bond"),
    ("EM HC Debt", 0.05, "Fidelity Asian Bond"),
    ("Private Markets", 0.05, "Global Infrastructure Income Fund"),
    ("Private Markets", 0.05, "Global Private Equity Fund"),
    ("Private Markets", 0.05, "Global Private Credit Fund"),
    ("Hedge Funds", 0.20, "Global Multi-Strategy Multi-PM"),
    ("Hedge Funds", 0.20, "Global Macro"),
]

# ── Look up metrics from existing dataframes ──────────────────

rows = []
for ac, taa_w, fname in selected:
    match = funds[funds["Fund"] == fname]
    if len(match) == 0:
        match = alts[alts["Fund"] == fname]
    if len(match) == 0:
        print(f"⚠ NOT FOUND: {fname}")
        continue
    f = match.iloc[0]
    rows.append({"AC": ac, "TAA_W": taa_w, "Fund": fname,
                  "Return": f["Return"], "Vol": f["Volatility"],
                  "Sharpe": f["Sharpe"], "AR": f["AR"]})

# Cash
rows.append({"AC": "Cash", "TAA_W": 0.05, "Fund": "Cash",
              "Return": rf_rate, "Vol": 0.001, "Sharpe": 0.0, "AR": 0.0})

df = pd.DataFrame(rows)

# ── Intra-class weights ───────────────────────────────────────

def sharpe_prop(sharpes):
    adj = np.maximum(sharpes, 0.01)
    return adj / adj.sum()

results = []
for ac in df["AC"].unique():
    grp = df[df["AC"] == ac]
    n = len(grp)
    taa_w = grp["TAA_W"].iloc[0]
    eq_w = np.ones(n) / n
    sp_w = sharpe_prop(grp["Sharpe"].values) if n > 1 else np.array([1.0])

    for i, (_, row) in enumerate(grp.iterrows()):
        results.append({**row.to_dict(),
            "EQ_port": eq_w[i] * taa_w, "SP_port": sp_w[i] * taa_w})

res = pd.DataFrame(results)

# ── Portfolio metrics ─────────────────────────────────────────

def port_metrics(w_col):
    w, r, v, ar = res[w_col].values, res["Return"].values, res["Vol"].values, res["AR"].values
    p_ret = w @ r
    p_vol = np.sqrt((w**2) @ (v**2))
    p_sharpe = (p_ret - rf_rate) / p_vol if p_vol > 0 else 0
    return p_ret, p_vol, p_sharpe, w @ ar

eq_ret, eq_vol, eq_sharpe, eq_war = port_metrics("EQ_port")
sp_ret, sp_vol, sp_sharpe, sp_war = port_metrics("SP_port")

# ── Output ────────────────────────────────────────────────────

print("=" * 80)
print("FUND SELECTION: EQUAL vs SHARPE-PROPORTIONAL")
print("=" * 80)

for ac in res["AC"].unique():
    grp = res[res["AC"] == ac]
    if ac == "Cash": continue
    print(f"\n▸ {ac} ({grp['EQ_port'].sum():.0%})")
    print(f"  {'Fund':45s} {'Sharpe':>7s}  {'Equal':>7s}  {'Sharpe-W':>8s}")
    print(f"  {'─'*70}")
    for _, row in grp.iterrows():
        print(f"  {row['Fund']:45s} {row['Sharpe']:7.3f}  {row['EQ_port']:6.1%}   {row['SP_port']:6.1%}")

print(f"\n{'='*80}")
print(f"PORTFOLIO COMPARISON (on ${total_investment/1e6:.0f}M)")
print(f"{'='*80}")
print(f"  {'Metric':25s} {'Equal':>12s} {'Sharpe-W':>12s} {'Delta':>10s}")
print(f"  {'─'*60}")
print(f"  {'Expected Return':25s} {eq_ret:11.2%}  {sp_ret:11.2%}  {sp_ret-eq_ret:+9.3%}")
print(f"  {'Volatility (approx)':25s} {eq_vol:11.2%}  {sp_vol:11.2%}  {sp_vol-eq_vol:+9.3%}")
print(f"  {'Sharpe Ratio':25s} {eq_sharpe:11.3f}  {sp_sharpe:11.3f}  {sp_sharpe-eq_sharpe:+9.3f}")
print(f"  {'Weighted AR':25s} {eq_war:11.1%}  {sp_war:11.1%}  {sp_war-eq_war:+9.1%}")
print(f"  {'─'*60}")
print(f"  {'Total Return ($)':25s} ${eq_ret*total_investment:>10,.0f}  ${sp_ret*total_investment:>10,.0f}  ${(sp_ret-eq_ret)*total_investment:>+9,.0f}")

diff_bps = (sp_ret - eq_ret) * 10000
print(f"\n  Difference: {diff_bps:+.0f} bps")
if abs(diff_bps) < 20:
    print(f"  → Marginal (<20bps). Equal weight recommended.")
elif abs(diff_bps) < 50:
    print(f"  → Modest (20-50bps). Sharpe-weight adds some value.")
else:
    print(f"  → Meaningful (>50bps). Sharpe-weight recommended.")

FUND SELECTION: EQUAL vs SHARPE-PROPORTIONAL

▸ US Eq (15%)
  Fund                                           Sharpe    Equal  Sharpe-W
  ──────────────────────────────────────────────────────────────────────
  AB American Growth Portfolio                    0.500    7.5%     6.3%
  JPMorgan America Equity Fund                    0.699    7.5%     8.7%

▸ EU Eq (5%)
  Fund                                           Sharpe    Equal  Sharpe-W
  ──────────────────────────────────────────────────────────────────────
  HSBC GIF Euroland Value Equity                  0.778    1.7%     1.9%
  JPMorgan Europe Equity Plus                     0.753    1.7%     1.8%
  Fidelity European Growth Fund                   0.541    1.7%     1.3%

▸ JP Eq (10%)
  Fund                                           Sharpe    Equal  Sharpe-W
  ──────────────────────────────────────────────────────────────────────
  M&G Japan                                       0.636    5.0%     3.5%
  Pictet Japanese Equity Oppo

In [22]:
# ═══════════════════════════════════════════════════════════════
# CELL: Fund Selection — Equal vs Sharpe-Proportional
# ═══════════════════════════════════════════════════════════════
# Dependencies: Cell 0-3 (funds, alts, Sharpe), TAA cell
# ═══════════════════════════════════════════════════════════════

total_investment = 45_000_000

selected = [
    ("US Eq", 0.12, "AB American Growth Portfolio"),
    ("US Eq", 0.12, "JPMorgan America Equity Fund"),
    ("EU Eq", 0.06, "HSBC GIF Euroland Value Equity"),
    ("EU Eq", 0.06, "JPMorgan Europe Equity Plus"),
    ("EU Eq", 0.06, "Fidelity European Growth Fund"),
    ("JP Eq", 0.10, "M&G Japan"),
    ("JP Eq", 0.10, "Pictet Japanese Equity Opportunities"),
    ("Asia ex-JP Eq", 0.07, "Schroder Asian Equity Yield"),
    ("Asia ex-JP Eq", 0.07, "Templeton Asian Smaller Companies"),
    ("Global IG", 0.25, "HSBC Global IG Securitised Credit"),
    ("Global IG", 0.25, "PIMCO Income"),
    ("Global IG", 0.25, "Schroder Global Credit Income"),
    ("Global HY", 0.05, "BlackRock Global High Yield Bond"),
    ("Global HY", 0.05, "Schroder Global High Yield"),
    ("EM HC Debt", 0.05, "BlackRock Emerging Markets Bond"),
    ("EM HC Debt", 0.05, "Fidelity Asian Bond"),
    ("Private Markets", 0.05, "Global Infrastructure Income Fund"),
    ("Private Markets", 0.05, "Global Private Equity Fund"),
    ("Private Markets", 0.05, "Global Private Credit Fund"),
    ("Hedge Funds", 0.20, "Global Multi-Strategy Multi-PM"),
    ("Hedge Funds", 0.20, "Global Macro"),
]

# ── Look up metrics from existing dataframes ──────────────────

rows = []
for ac, taa_w, fname in selected:
    match = funds[funds["Fund"] == fname]
    if len(match) == 0:
        match = alts[alts["Fund"] == fname]
    if len(match) == 0:
        print(f"⚠ NOT FOUND: {fname}")
        continue
    f = match.iloc[0]
    rows.append({"AC": ac, "TAA_W": taa_w, "Fund": fname,
                  "Return": f["Return"], "Vol": f["Volatility"],
                  "Sharpe": f["Sharpe"], "AR": f["AR"]})

# Cash
rows.append({"AC": "Cash", "TAA_W": 0.05, "Fund": "Cash",
              "Return": rf_rate, "Vol": 0.001, "Sharpe": 0.0, "AR": 0.0})

df = pd.DataFrame(rows)

# ── Intra-class weights ───────────────────────────────────────

def sharpe_prop(sharpes):
    adj = np.maximum(sharpes, 0.01)
    return adj / adj.sum()

results = []
for ac in df["AC"].unique():
    grp = df[df["AC"] == ac]
    n = len(grp)
    taa_w = grp["TAA_W"].iloc[0]
    eq_w = np.ones(n) / n
    sp_w = sharpe_prop(grp["Sharpe"].values) if n > 1 else np.array([1.0])

    for i, (_, row) in enumerate(grp.iterrows()):
        results.append({**row.to_dict(),
            "EQ_port": eq_w[i] * taa_w, "SP_port": sp_w[i] * taa_w})

res = pd.DataFrame(results)

# ── Portfolio metrics ─────────────────────────────────────────

def port_metrics(w_col):
    w, r, v, ar = res[w_col].values, res["Return"].values, res["Vol"].values, res["AR"].values
    p_ret = w @ r
    p_vol = np.sqrt((w**2) @ (v**2))
    p_sharpe = (p_ret - rf_rate) / p_vol if p_vol > 0 else 0
    return p_ret, p_vol, p_sharpe, w @ ar

eq_ret, eq_vol, eq_sharpe, eq_war = port_metrics("EQ_port")
sp_ret, sp_vol, sp_sharpe, sp_war = port_metrics("SP_port")

# ── Output ────────────────────────────────────────────────────

print("=" * 80)
print("FUND SELECTION: EQUAL vs SHARPE-PROPORTIONAL")
print("=" * 80)

for ac in res["AC"].unique():
    grp = res[res["AC"] == ac]
    if ac == "Cash": continue
    print(f"\n▸ {ac} ({grp['EQ_port'].sum():.0%})")
    print(f"  {'Fund':45s} {'Sharpe':>7s}  {'Equal':>7s}  {'Sharpe-W':>8s}")
    print(f"  {'─'*70}")
    for _, row in grp.iterrows():
        print(f"  {row['Fund']:45s} {row['Sharpe']:7.3f}  {row['EQ_port']:6.1%}   {row['SP_port']:6.1%}")

print(f"\n{'='*80}")
print(f"PORTFOLIO COMPARISON (on ${total_investment/1e6:.0f}M)")
print(f"{'='*80}")
print(f"  {'Metric':25s} {'Equal':>12s} {'Sharpe-W':>12s} {'Delta':>10s}")
print(f"  {'─'*60}")
print(f"  {'Expected Return':25s} {eq_ret:11.2%}  {sp_ret:11.2%}  {sp_ret-eq_ret:+9.3%}")
print(f"  {'Volatility (approx)':25s} {eq_vol:11.2%}  {sp_vol:11.2%}  {sp_vol-eq_vol:+9.3%}")
print(f"  {'Sharpe Ratio':25s} {eq_sharpe:11.3f}  {sp_sharpe:11.3f}  {sp_sharpe-eq_sharpe:+9.3f}")
print(f"  {'Weighted AR':25s} {eq_war:11.1%}  {sp_war:11.1%}  {sp_war-eq_war:+9.1%}")
print(f"  {'─'*60}")
print(f"  {'Total Return ($)':25s} ${eq_ret*total_investment:>10,.0f}  ${sp_ret*total_investment:>10,.0f}  ${(sp_ret-eq_ret)*total_investment:>+9,.0f}")

diff_bps = (sp_ret - eq_ret) * 10000
print(f"\n  Difference: {diff_bps:+.0f} bps")
if abs(diff_bps) < 20:
    print(f"  → Marginal (<20bps). Equal weight recommended.")
elif abs(diff_bps) < 50:
    print(f"  → Modest (20-50bps). Sharpe-weight adds some value.")
else:
    print(f"  → Meaningful (>50bps). Sharpe-weight recommended.")

FUND SELECTION: EQUAL vs SHARPE-PROPORTIONAL

▸ US Eq (12%)
  Fund                                           Sharpe    Equal  Sharpe-W
  ──────────────────────────────────────────────────────────────────────
  AB American Growth Portfolio                    0.500    6.0%     5.0%
  JPMorgan America Equity Fund                    0.699    6.0%     7.0%

▸ EU Eq (6%)
  Fund                                           Sharpe    Equal  Sharpe-W
  ──────────────────────────────────────────────────────────────────────
  HSBC GIF Euroland Value Equity                  0.778    2.0%     2.3%
  JPMorgan Europe Equity Plus                     0.753    2.0%     2.2%
  Fidelity European Growth Fund                   0.541    2.0%     1.6%

▸ JP Eq (10%)
  Fund                                           Sharpe    Equal  Sharpe-W
  ──────────────────────────────────────────────────────────────────────
  M&G Japan                                       0.636    5.0%     3.5%
  Pictet Japanese Equity Oppo

In [6]:
print("=" * 60)
print("CORE FORMULAS")
print("=" * 60)
print(f"  {'Portfolio Return':26s}  Rp = Sum( wi * Ri )")
print(f"  {'Portfolio Volatility':26s}  sig_p = sqrt( w' Sigma w )")
print(f"  {'Sharpe Ratio':26s}  SR = (Rp - rf) / sig_p")
print(f"  {'Cash Return (Yield)':26s}  Yp = Sum( wi * Yield_i )")
print(f"  {'Capital Gain':26s}  CGp = Rp - Yp")
print(f"  {'Weighted AR':26s}  WAR = Sum( wi * ARi )")
print(f"  {'Net Return':26s}  Net = R_gross * P - B * c")
print(f"  {'Return on Equity':26s}  ROE = Net Return / Equity")
print(f"  {'AR-Adjusted Return':26s}  R_adj = (R - AR*c) / (1 - AR)")
print(f"  {'AR-Adjusted Vol':26s}  sig_adj = sig / (1 - AR)")
print("=" * 60)

CORE FORMULAS
  Portfolio Return            Rp = Sum( wi * Ri )
  Portfolio Volatility        sig_p = sqrt( w' Sigma w )
  Sharpe Ratio                SR = (Rp - rf) / sig_p
  Cash Return (Yield)         Yp = Sum( wi * Yield_i )
  Capital Gain                CGp = Rp - Yp
  Weighted AR                 WAR = Sum( wi * ARi )
  Net Return                  Net = R_gross * P - B * c
  Return on Equity            ROE = Net Return / Equity
  AR-Adjusted Return          R_adj = (R - AR*c) / (1 - AR)
  AR-Adjusted Vol             sig_adj = sig / (1 - AR)


In [8]:
print("=" * 60)
print("DATA SOURCES & ASSUMPTIONS")
print("=" * 60)
print("  Corr Matrix   WRDS/CRSP monthly (Feb21-Dec24)")
print("  Fund Data     Appendix I (44) + II (9 alts)")
print("  rf = 3.50%  |  Borrow = 5.00%  |  Target = $80M")
print("=" * 60)

DATA SOURCES & ASSUMPTIONS
  Corr Matrix   WRDS/CRSP monthly (Feb21-Dec24)
  Fund Data     Appendix I (44) + II (9 alts)
  rf = 3.50%  |  Borrow = 5.00%  |  Target = $80M


In [9]:
print("=" * 60)
print("SAA OPTIMIZATION SUMMARY")
print("=" * 60)
print("  Grid search: 11 asset classes x weight ranges")
print("  Total combinations:  3,072")
print("  Valid (sum = 100%):  202")
print("-" * 60)
print("  #1 by Sharpe:")
print("    Return=11.26%  Vol=7.53%  Sharpe=1.030")
print("    US15 EU5 JP10 AxJ10 IG20 HY5 EMHC5 PM5 HF20 Cash5")
print("=" * 60)

print()

print("=" * 60)
print("SAA vs TAA COMPARISON")
print("=" * 60)
print(f"  {'Asset Class':18s} {'SAA':>5s} {'TAA':>5s} {'Tilt':>6s} {'AR':>5s}")
print("-" * 60)
print(f"  {'US Eq':18s} {'15%':>5s} {'12%':>5s} {'-3%':>6s} {'70%':>5s}")
print(f"  {'EU Eq':18s} {'5%':>5s} {'6%':>5s} {'+1%':>6s} {'70%':>5s}")
print(f"  {'JP Eq':18s} {'10%':>5s} {'10%':>5s} {'--':>6s} {'70%':>5s}")
print(f"  {'Asia ex-JP Eq':18s} {'10%':>5s} {'7%':>5s} {'-3%':>6s} {'70%':>5s}")
print(f"  {'Global IG':18s} {'20%':>5s} {'25%':>5s} {'+5%':>6s} {'80%':>5s}")
print(f"  {'Global HY':18s} {'5%':>5s} {'5%':>5s} {'--':>6s} {'60%':>5s}")
print(f"  {'EM HC Debt':18s} {'5%':>5s} {'5%':>5s} {'--':>6s} {'75%':>5s}")
print(f"  {'Private Markets':18s} {'5%':>5s} {'5%':>5s} {'--':>6s} {'50%':>5s}")
print(f"  {'Hedge Funds':18s} {'20%':>5s} {'20%':>5s} {'--':>6s} {'40%':>5s}")
print(f"  {'Cash':18s} {'5%':>5s} {'5%':>5s} {'--':>6s} {'0%':>5s}")
print("-" * 60)
print(f"  {'Metric':25s} {'SAA':>8s} {'TAA':>8s} {'Delta':>8s}")
print("-" * 60)
print(f"  {'Expected Return':25s} {'11.26%':>8s} {'10.90%':>8s} {'-0.36%':>8s}")
print(f"  {'Volatility':25s} {'7.53%':>8s} {'7.01%':>8s} {'-0.53%':>8s}")
print(f"  {'Sharpe Ratio':25s} {'1.030':>8s} {'1.057':>8s} {'+0.027':>8s}")
print(f"  {'Weighted AR':25s} {'61.2%':>8s} {'61.7%':>8s} {'+0.5%':>8s}")
print("=" * 60)

SAA OPTIMIZATION SUMMARY
  Grid search: 11 asset classes x weight ranges
  Total combinations:  3,072
  Valid (sum = 100%):  202
------------------------------------------------------------
  #1 by Sharpe:
    Return=11.26%  Vol=7.53%  Sharpe=1.030
    US15 EU5 JP10 AxJ10 IG20 HY5 EMHC5 PM5 HF20 Cash5

SAA vs TAA COMPARISON
  Asset Class          SAA   TAA   Tilt    AR
------------------------------------------------------------
  US Eq                15%   12%    -3%   70%
  EU Eq                 5%    6%    +1%   70%
  JP Eq                10%   10%     --   70%
  Asia ex-JP Eq        10%    7%    -3%   70%
  Global IG            20%   25%    +5%   80%
  Global HY             5%    5%     --   60%
  EM HC Debt            5%    5%     --   75%
  Private Markets       5%    5%     --   50%
  Hedge Funds          20%   20%     --   40%
  Cash                  5%    5%     --    0%
------------------------------------------------------------
  Metric                         SAA      TAA 

In [19]:
# 셀 2: TAA
print("=" * 50)
print("TAA ADJUSTMENT (from SAA)")
print("=" * 50)
print(f"  {'Asset Class':18s} {'SAA':>6s} {'TAA':>6s} {'Tilt':>6s}")
print("-" * 50)
print(f"  {'US Eq':18s} {'15%':>6s} {'12%':>6s} {'-3%':>6s}")
print(f"  {'EU Eq':18s} {'5%':>6s} {'6%':>6s} {'+1%':>6s}")
print(f"  {'Asia ex-JP Eq':18s} {'10%':>6s} {'7%':>6s} {'-3%':>6s}")
print(f"  {'Global IG':18s} {'20%':>6s} {'25%':>6s} {'+5%':>6s}")
print("-" * 50)
print(f"  {'Return':18s} {'11.26%':>6s} {'10.90%':>8s}")
print(f"  {'Volatility':18s} {'7.53%':>6s} {'7.01%':>8s}")
print(f"  {'Sharpe':18s} {'1.030':>6s} {'1.057':>8s}")
print("=" * 50)

TAA ADJUSTMENT (from SAA)
  Asset Class           SAA    TAA   Tilt
--------------------------------------------------
  US Eq                 15%    12%    -3%
  EU Eq                  5%     6%    +1%
  Asia ex-JP Eq         10%     7%    -3%
  Global IG             20%    25%    +5%
--------------------------------------------------
  Return             11.26%   10.90%
  Volatility          7.53%    7.01%
  Sharpe              1.030    1.057


In [22]:

print("  Grid search: 3,072 -> 202 valid")
print("  #1 by Sharpe:")
print("-" * 50)
print(f"  {'Asset Class':18s} {'Weight':>8s} {'AR':>6s}")
print("-" * 50)
print(f"  {'US Eq':18s} {'15%':>8s} {'70%':>6s}")
print(f"  {'EU Eq':18s} {'5%':>8s} {'70%':>6s}")
print(f"  {'JP Eq':18s} {'10%':>8s} {'70%':>6s}")
print(f"  {'Asia ex-JP Eq':18s} {'10%':>8s} {'70%':>6s}")
print(f"  {'Global IG':18s} {'20%':>8s} {'80%':>6s}")
print(f"  {'Global HY':18s} {'5%':>8s} {'60%':>6s}")
print(f"  {'EM HC Debt':18s} {'5%':>8s} {'75%':>6s}")
print(f"  {'Private Markets':18s} {'5%':>8s} {'50%':>6s}")
print(f"  {'Hedge Funds':18s} {'20%':>8s} {'40%':>6s}")
print(f"  {'Cash':18s} {'5%':>8s} {'0%':>6s}")
print("-" * 50)
print(f"  Expected Return:    11.26%")
print(f"  Volatility:          7.53%")
print(f"  Sharpe Ratio:        1.030")
print(f"  Weighted AR:        61.2%")
print("=" * 50)

  Grid search: 3,072 -> 202 valid
  #1 by Sharpe:
--------------------------------------------------
  Asset Class          Weight     AR
--------------------------------------------------
  US Eq                   15%    70%
  EU Eq                    5%    70%
  JP Eq                   10%    70%
  Asia ex-JP Eq           10%    70%
  Global IG               20%    80%
  Global HY                5%    60%
  EM HC Debt               5%    75%
  Private Markets          5%    50%
  Hedge Funds             20%    40%
  Cash                     5%     0%
--------------------------------------------------
  Expected Return:    11.26%
  Volatility:          7.53%
  Sharpe Ratio:        1.030
  Weighted AR:        61.2%


In [23]:

print("  Macro-driven tilts on SAA baseline")
print("  Eq -5%, FI +5%, Alts unchanged")
print("-" * 50)
print(f"  {'Asset Class':18s} {'Weight':>8s} {'AR':>6s}")
print("-" * 50)
print(f"  {'US Eq':18s} {'12%':>8s} {'70%':>6s}")
print(f"  {'EU Eq':18s} {'6%':>8s} {'70%':>6s}")
print(f"  {'JP Eq':18s} {'10%':>8s} {'70%':>6s}")
print(f"  {'Asia ex-JP Eq':18s} {'7%':>8s} {'70%':>6s}")
print(f"  {'Global IG':18s} {'25%':>8s} {'80%':>6s}")
print(f"  {'Global HY':18s} {'5%':>8s} {'60%':>6s}")
print(f"  {'EM HC Debt':18s} {'5%':>8s} {'75%':>6s}")
print(f"  {'Private Markets':18s} {'5%':>8s} {'50%':>6s}")
print(f"  {'Hedge Funds':18s} {'20%':>8s} {'40%':>6s}")
print(f"  {'Cash':18s} {'5%':>8s} {'0%':>6s}")
print("-" * 50)
print(f"  Expected Return:    10.90%")
print(f"  Volatility:          7.01%")
print(f"  Sharpe Ratio:        1.057")
print(f"  Weighted AR:        61.7%")
print("=" * 50)

  Macro-driven tilts on SAA baseline
  Eq -5%, FI +5%, Alts unchanged
--------------------------------------------------
  Asset Class          Weight     AR
--------------------------------------------------
  US Eq                   12%    70%
  EU Eq                    6%    70%
  JP Eq                   10%    70%
  Asia ex-JP Eq            7%    70%
  Global IG               25%    80%
  Global HY                5%    60%
  EM HC Debt               5%    75%
  Private Markets          5%    50%
  Hedge Funds             20%    40%
  Cash                     5%     0%
--------------------------------------------------
  Expected Return:    10.90%
  Volatility:          7.01%
  Sharpe Ratio:        1.057
  Weighted AR:        61.7%


In [32]:
# 셀 1: 좌 - Equity + IG
print("=" * 55)
print("FUND SELECTION: EQUITY (9 of 20)")
print("=" * 55)
print("  Sharpe-proportional weight within class")
print("-" * 55)
print(f"  {'Fund':24s} {'SAA':>6s} {'TAA':>6s}")
print("-" * 55)
print("  US Eq")
print(f"    {'AB American Growth':22s} {'6.3%':>6s} {'5.0%':>6s}")
print(f"    {'JPMorgan America Eq':22s} {'8.7%':>6s} {'7.0%':>6s}")
print("  EU Eq")
print(f"    {'HSBC Euroland Value':22s} {'1.9%':>6s} {'2.3%':>6s}")
print(f"    {'JPMorgan Europe Plus':22s} {'1.8%':>6s} {'2.2%':>6s}")
print(f"    {'Fidelity EU Growth':22s} {'1.3%':>6s} {'1.6%':>6s}")
print("  JP Eq")
print(f"    {'M&G Japan':22s} {'3.5%':>6s} {'3.5%':>6s}")
print(f"    {'Pictet JP Eq Opp':22s} {'6.5%':>6s} {'6.5%':>6s}")
print("  Asia ex-JP")
print(f"    {'Schroder Asian Yld':22s} {'4.8%':>6s} {'3.3%':>6s}")
print(f"    {'Templeton Asian SC':22s} {'5.2%':>6s} {'3.7%':>6s}")
print("  Fixed Income")
print(f"    {'HSBC Glb IG Sec':22s} {'6.6%':>6s} {'8.3%':>6s}")
print("-" * 55)
print("  Excluded: Thematic (10), EM LC (3)")
print("=" * 55)

FUND SELECTION: EQUITY (9 of 20)
  Sharpe-proportional weight within class
-------------------------------------------------------
  Fund                        SAA    TAA
-------------------------------------------------------
  US Eq
    AB American Growth       6.3%   5.0%
    JPMorgan America Eq      8.7%   7.0%
  EU Eq
    HSBC Euroland Value      1.9%   2.3%
    JPMorgan Europe Plus     1.8%   2.2%
    Fidelity EU Growth       1.3%   1.6%
  JP Eq
    M&G Japan                3.5%   3.5%
    Pictet JP Eq Opp         6.5%   6.5%
  Asia ex-JP
    Schroder Asian Yld       4.8%   3.3%
    Templeton Asian SC       5.2%   3.7%
  Fixed Income
    HSBC Glb IG Sec          6.6%   8.3%
-------------------------------------------------------
  Excluded: Thematic (10), EM LC (3)


In [33]:
# 셀 1: 좌 - Equity
print("=" * 55)
print("FUND SELECTION: EQUITY (9 of 20)")
print("=" * 55)
print("  Top Sharpe per class, equal weight within class")
print("-" * 55)
print(f"  {'Fund':24s} {'Ret':>6s} {'Vol':>6s} {'SR':>6s} {'Wt':>6s}")
print("-" * 55)
print("  US Eq (2 of 5)")
print(f"    {'AB American Growth':22s} {'12.6%':>6s} {'18.2%':>6s} {'0.50':>6s} {'6.0%':>6s}")
print(f"    {'JPMorgan America Eq':22s} {'14.4%':>6s} {'15.6%':>6s} {'0.70':>6s} {'6.0%':>6s}")
print("  EU Eq (3 of 5)")
print(f"    {'HSBC Euroland Value':22s} {'16.5%':>6s} {'16.7%':>6s} {'0.78':>6s} {'2.0%':>6s}")
print(f"    {'JPMorgan Europe Plus':22s} {'15.7%':>6s} {'16.2%':>6s} {'0.75':>6s} {'2.0%':>6s}")
print(f"    {'Fidelity EU Growth':22s} {'10.1%':>6s} {'12.2%':>6s} {'0.54':>6s} {'2.0%':>6s}")
print("  JP Eq (2 of 5)")
print(f"    {'Pictet JP Eq Opp':22s} {'18.0%':>6s} {'12.4%':>6s} {'1.17':>6s} {'5.0%':>6s}")
print(f"    {'M&G Japan':22s} {'12.6%':>6s} {'14.3%':>6s} {'0.64':>6s} {'5.0%':>6s}")
print("  Asia ex-JP (2 of 5)")
print(f"    {'Templeton Asian SC':22s} {'8.8%':>6s} {'15.5%':>6s} {'0.35':>6s} {'3.5%':>6s}")
print(f"    {'Schroder Asian Yld':22s} {'8.7%':>6s} {'16.7%':>6s} {'0.31':>6s} {'3.5%':>6s}")
print("-" * 55)
print("  Excluded: Thematic (10), EM LC Debt (3)")
print("  Sharpe-prop vs equal: +13bps, marginal")
print("=" * 55)

FUND SELECTION: EQUITY (9 of 20)
  Top Sharpe per class, equal weight within class
-------------------------------------------------------
  Fund                        Ret    Vol     SR     Wt
-------------------------------------------------------
  US Eq (2 of 5)
    AB American Growth      12.6%  18.2%   0.50   6.0%
    JPMorgan America Eq     14.4%  15.6%   0.70   6.0%
  EU Eq (3 of 5)
    HSBC Euroland Value     16.5%  16.7%   0.78   2.0%
    JPMorgan Europe Plus    15.7%  16.2%   0.75   2.0%
    Fidelity EU Growth      10.1%  12.2%   0.54   2.0%
  JP Eq (2 of 5)
    Pictet JP Eq Opp        18.0%  12.4%   1.17   5.0%
    M&G Japan               12.6%  14.3%   0.64   5.0%
  Asia ex-JP (2 of 5)
    Templeton Asian SC       8.8%  15.5%   0.35   3.5%
    Schroder Asian Yld       8.7%  16.7%   0.31   3.5%
-------------------------------------------------------
  Excluded: Thematic (10), EM LC Debt (3)
  Sharpe-prop vs equal: +13bps, marginal


In [34]:
# 셀 1: 좌 - Equity
print("=" * 58)
print("FUND SELECTION: EQUITY (9 of 20)")
print("=" * 58)
print("  Top Sharpe per class, equal weight within class")
print("-" * 58)
print(f"  {'Fund':22s} {'Ret':>6s} {'Vol':>6s} {'SR':>6s} {'SAA':>6s} {'TAA':>6s}")
print("-" * 58)
print("  US Eq (2 of 5)")
print(f"    {'AB American Growth':20s} {'12.6%':>6s} {'18.2%':>6s} {'0.50':>6s} {'6.3%':>6s} {'5.0%':>6s}")
print(f"    {'JPMorgan America Eq':20s} {'14.4%':>6s} {'15.6%':>6s} {'0.70':>6s} {'8.7%':>6s} {'7.0%':>6s}")
print("  EU Eq (3 of 5)")
print(f"    {'HSBC Euroland Value':20s} {'16.5%':>6s} {'16.7%':>6s} {'0.78':>6s} {'1.9%':>6s} {'2.3%':>6s}")
print(f"    {'JPMorgan Europe Plus':20s} {'15.7%':>6s} {'16.2%':>6s} {'0.75':>6s} {'1.8%':>6s} {'2.2%':>6s}")
print(f"    {'Fidelity EU Growth':20s} {'10.1%':>6s} {'12.2%':>6s} {'0.54':>6s} {'1.3%':>6s} {'1.6%':>6s}")
print("  JP Eq (2 of 5)")
print(f"    {'Pictet JP Eq Opp':20s} {'18.0%':>6s} {'12.4%':>6s} {'1.17':>6s} {'6.5%':>6s} {'6.5%':>6s}")
print(f"    {'M&G Japan':20s} {'12.6%':>6s} {'14.3%':>6s} {'0.64':>6s} {'3.5%':>6s} {'3.5%':>6s}")
print("  Asia ex-JP (2 of 5)")
print(f"    {'Templeton Asian SC':20s} {'8.8%':>6s} {'15.5%':>6s} {'0.35':>6s} {'5.2%':>6s} {'3.7%':>6s}")
print(f"    {'Schroder Asian Yld':20s} {'8.7%':>6s} {'16.7%':>6s} {'0.31':>6s} {'4.8%':>6s} {'3.3%':>6s}")
print("-" * 58)
print("  Excluded: Thematic (10), EM LC Debt (3)")
print("  Sharpe-prop vs equal: +13bps, marginal")
print("=" * 58)

FUND SELECTION: EQUITY (9 of 20)
  Top Sharpe per class, equal weight within class
----------------------------------------------------------
  Fund                      Ret    Vol     SR    SAA    TAA
----------------------------------------------------------
  US Eq (2 of 5)
    AB American Growth    12.6%  18.2%   0.50   6.3%   5.0%
    JPMorgan America Eq   14.4%  15.6%   0.70   8.7%   7.0%
  EU Eq (3 of 5)
    HSBC Euroland Value   16.5%  16.7%   0.78   1.9%   2.3%
    JPMorgan Europe Plus  15.7%  16.2%   0.75   1.8%   2.2%
    Fidelity EU Growth    10.1%  12.2%   0.54   1.3%   1.6%
  JP Eq (2 of 5)
    Pictet JP Eq Opp      18.0%  12.4%   1.17   6.5%   6.5%
    M&G Japan             12.6%  14.3%   0.64   3.5%   3.5%
  Asia ex-JP (2 of 5)
    Templeton Asian SC     8.8%  15.5%   0.35   5.2%   3.7%
    Schroder Asian Yld     8.7%  16.7%   0.31   4.8%   3.3%
----------------------------------------------------------
  Excluded: Thematic (10), EM LC Debt (3)
  Sharpe-prop vs equal: +

In [35]:
# 셀 2: 우 - FI + Alts
print("=" * 58)
print("FUND SELECTION: FI + ALTS (12 of 33)")
print("=" * 58)
print("  Top Sharpe per class, equal weight within class")
print("-" * 58)
print(f"  {'Fund':22s} {'Ret':>6s} {'Vol':>6s} {'SR':>6s} {'SAA':>6s} {'TAA':>6s}")
print("-" * 58)
print("  Global IG (3 of 4)")
print(f"    {'HSBC Glb IG Sec':20s} {'3.5%':>6s} {'2.0%':>6s} {'0.00':>6s} {'6.6%':>6s} {'8.3%':>6s}")
print(f"    {'PIMCO Income':20s} {'3.2%':>6s} {'5.7%':>6s} {'-0.1':>6s} {'6.7%':>6s} {'8.3%':>6s}")
print(f"    {'Schroder Glb Credit':20s} {'2.4%':>6s} {'6.3%':>6s} {'-0.2':>6s} {'6.7%':>6s} {'8.3%':>6s}")
print("  Global HY (2 of 4)")
print(f"    {'BR Global HY Bond':20s} {'3.4%':>6s} {'6.7%':>6s} {'-0.0':>6s} {'0.9%':>6s} {'0.9%':>6s}")
print(f"    {'Schroder Global HY':20s} {'3.8%':>6s} {'6.9%':>6s} {'0.04':>6s} {'4.1%':>6s} {'4.1%':>6s}")
print("  EM HC Debt (2 of 3)")
print(f"    {'BR EM Bond':20s} {'3.2%':>6s} {'9.8%':>6s} {'-0.0':>6s} {'2.5%':>6s} {'2.5%':>6s}")
print(f"    {'Fidelity Asian Bond':20s} {'1.0%':>6s} {'9.8%':>6s} {'-0.3':>6s} {'2.5%':>6s} {'2.5%':>6s}")
print("  Private Markets (3 of 4)")
print(f"    {'Glb Infra Income':20s} {'8.0%':>6s} {'11.0%':>6s} {'0.41':>6s} {'1.9%':>6s} {'1.9%':>6s}")
print(f"    {'Glb Private Equity':20s} {'9.9%':>6s} {'19.6%':>6s} {'0.33':>6s} {'1.5%':>6s} {'1.5%':>6s}")
print(f"    {'Glb Private Credit':20s} {'8.2%':>6s} {'13.6%':>6s} {'0.35':>6s} {'1.6%':>6s} {'1.6%':>6s}")
print("  Hedge Funds (2 of 5)")
print(f"    {'Glb Multi-Strat PM':20s} {'10.3%':>6s} {'6.8%':>6s} {'0.78':>6s} {'9.9%':>6s} {'9.9%':>6s}")
print(f"    {'Global Macro':20s} {'9.4%':>6s} {'5.8%':>6s} {'1.02':>6s} {'10.1%':>6s} {'10.1%':>6s}")
print("=" * 58)

FUND SELECTION: FI + ALTS (12 of 33)
  Top Sharpe per class, equal weight within class
----------------------------------------------------------
  Fund                      Ret    Vol     SR    SAA    TAA
----------------------------------------------------------
  Global IG (3 of 4)
    HSBC Glb IG Sec        3.5%   2.0%   0.00   6.6%   8.3%
    PIMCO Income           3.2%   5.7%   -0.1   6.7%   8.3%
    Schroder Glb Credit    2.4%   6.3%   -0.2   6.7%   8.3%
  Global HY (2 of 4)
    BR Global HY Bond      3.4%   6.7%   -0.0   0.9%   0.9%
    Schroder Global HY     3.8%   6.9%   0.04   4.1%   4.1%
  EM HC Debt (2 of 3)
    BR EM Bond             3.2%   9.8%   -0.0   2.5%   2.5%
    Fidelity Asian Bond    1.0%   9.8%   -0.3   2.5%   2.5%
  Private Markets (3 of 4)
    Glb Infra Income       8.0%  11.0%   0.41   1.9%   1.9%
    Glb Private Equity     9.9%  19.6%   0.33   1.5%   1.5%
    Glb Private Credit     8.2%  13.6%   0.35   1.6%   1.6%
  Hedge Funds (2 of 5)
    Glb Multi-Strat P

In [36]:
# 셀 1: 좌 - Funding Structure + AR Breakdown
print("=" * 55)
print("CREDIT SOLUTION: FUNDING STRUCTURE")
print("=" * 55)
print("  Client cash:            $45.0M")
print("  Fund AR borrowing:      $35.0M")
print("  Total portfolio:        $80.0M")
print("  Leverage:               1.78x (on $45M equity)")
print("  External Lombard:       $0M (capacity $26.25M)")
print("-" * 55)
print(f"  {'Asset Class':16s} {'Wt':>5s} {'Alloc':>8s} {'AR':>5s} {'AR Cap':>8s}")
print("-" * 55)
print(f"  {'US Eq':16s} {'12%':>5s} {'$9.6M':>8s} {'70%':>5s} {'$6.7M':>8s}")
print(f"  {'EU Eq':16s} {'6%':>5s} {'$4.8M':>8s} {'70%':>5s} {'$3.4M':>8s}")
print(f"  {'JP Eq':16s} {'10%':>5s} {'$8.0M':>8s} {'70%':>5s} {'$5.6M':>8s}")
print(f"  {'Asia ex-JP Eq':16s} {'7%':>5s} {'$5.6M':>8s} {'70%':>5s} {'$3.9M':>8s}")
print(f"  {'Global IG':16s} {'25%':>5s} {'$20.0M':>8s} {'80%':>5s} {'$16.0M':>8s}")
print(f"  {'Global HY':16s} {'5%':>5s} {'$4.0M':>8s} {'60%':>5s} {'$2.4M':>8s}")
print(f"  {'EM HC Debt':16s} {'5%':>5s} {'$4.0M':>8s} {'75%':>5s} {'$3.0M':>8s}")
print(f"  {'Private Mkt':16s} {'5%':>5s} {'$4.0M':>8s} {'50%':>5s} {'$2.0M':>8s}")
print(f"  {'Hedge Funds':16s} {'20%':>5s} {'$16.0M':>8s} {'40%':>5s} {'$6.4M':>8s}")
print(f"  {'Cash':16s} {'5%':>5s} {'$4.0M':>8s} {'0%':>5s} {'$0.0M':>8s}")
print("-" * 55)
print(f"  {'Total':16s} {'100%':>5s} {'$80.0M':>8s} {'62%':>5s} {'$49.4M':>8s}")
print("=" * 55)

CREDIT SOLUTION: FUNDING STRUCTURE
  Client cash:            $45.0M
  Fund AR borrowing:      $35.0M
  Total portfolio:        $80.0M
  Leverage:               1.78x (on $45M equity)
  External Lombard:       $0M (capacity $26.25M)
-------------------------------------------------------
  Asset Class         Wt    Alloc    AR   AR Cap
-------------------------------------------------------
  US Eq              12%    $9.6M   70%    $6.7M
  EU Eq               6%    $4.8M   70%    $3.4M
  JP Eq              10%    $8.0M   70%    $5.6M
  Asia ex-JP Eq       7%    $5.6M   70%    $3.9M
  Global IG          25%   $20.0M   80%   $16.0M
  Global HY           5%    $4.0M   60%    $2.4M
  EM HC Debt          5%    $4.0M   75%    $3.0M
  Private Mkt         5%    $4.0M   50%    $2.0M
  Hedge Funds        20%   $16.0M   40%    $6.4M
  Cash                5%    $4.0M    0%    $0.0M
-------------------------------------------------------
  Total             100%   $80.0M   62%   $49.4M


In [38]:
# 셀 2: 우 - Return Calculation
print("=" * 55)
print("CREDIT SOLUTION: RETURN CALCULATION")
print("=" * 55)
print("  Gross portfolio return:     10.90%  = $8.72M")
print("  Gross income (yield):        2.81%  = $2.25M")
print("-" * 55)
print("  Interest - Fund AR:         $1.75M")
print("    ($35.0M x 5.0% SOFR+75bps)")
print("  Interest - Lombard:         $0.00M")
print("  Total annual interest:      $1.75M")
print("-" * 55)
print("  Net return:                 $6.97M")
print("  Net return on equity:       15.50%  (on $45M)")
print("  Net income after interest:  $0.50M")
print("-" * 55)
print("  Income target:              $300K")
print("  Income coverage:            1.7x  -> MET")
print("-" * 55)
print("  SUMMARY")
print("-" * 55)
print("  Portfolio:  $80M   |  Equity:  $45M")
print("  Borrowing:  $35M   |  Cost:    5.0%")
print("  Gross ret:  10.90% |  Net:     15.50% ROE")
print("  Wtd AR:     61.7%  |  Income:  1.7x target")
print("=" * 55)

CREDIT SOLUTION: RETURN CALCULATION
  Gross portfolio return:     10.90%  = $8.72M
  Gross income (yield):        2.81%  = $2.25M
-------------------------------------------------------
  Interest - Fund AR:         $1.75M
    ($35.0M x 5.0% SOFR+75bps)
  Interest - Lombard:         $0.00M
  Total annual interest:      $1.75M
-------------------------------------------------------
  Net return:                 $6.97M
  Net return on equity:       15.50%  (on $45M)
  Net income after interest:  $0.50M
-------------------------------------------------------
  Income target:              $300K
  Income coverage:            1.7x  -> MET
-------------------------------------------------------
  SUMMARY
-------------------------------------------------------
  Portfolio:  $80M   |  Equity:  $45M
  Borrowing:  $35M   |  Cost:    5.0%
  Gross ret:  10.90% |  Net:     15.50% ROE
  Wtd AR:     61.7%  |  Income:  1.7x target
